<a href="https://colab.research.google.com/github/phillip-jaeslee/PULSIM/blob/main/PULSIM_extension.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PULSIM — extension modules

Four modules beyond the worked examples of the main notebook (`PULSIM_colab.ipynb`), with the inputs of a real experiment: field, chemical shifts in ppm, the frequency of each pulse.

| Cell | What it does |
|---|---|
| Install / Load | Clone PULSIM, install dependencies, load the tools (run once per session) |
| **1. Water suppression** | Hard-pulse trains (jump-return, binomial of any order) and WATERGATE with its gradient echo, against the ¹H chemical shift |
| **2. Bloch–Siegert shift** | The phase a sequence of selective pulses leaves on spins it does not excite, and how to correct it over a band |
| **3. Relaxation-aware delays** | Delay optimization with T₁ and T₂ for each spin: the best delay, and the best delay against T₂ |
| **4. Coherence selection with gradients** | Gradients in a multiple-spin sequence: which coherence pathways survive (gradient echo, heteronuclear echo/antiecho, double-quantum filter), and the signal against the gradient ratio |

**How to use:** `Runtime` → `Run all`, then pick a **preset**, edit the rows and press **Run**; **Reload preset** restores the preset. **compare** overlays several values of one parameter; **figures** saves the run's figures (PNG, PDF or SVG) and **data** its numbers (CSV).

Units: chemical shifts in **ppm**, field in **MHz** (¹H frequency), durations in **ms**, couplings in **Hz**, offsets in the multiple-spin widget in **rad/ms**.

Please cite: <https://github.com/phillip-jaeslee/PULSIM>

In [ ]:
#@title Install PULSIM and load the tools { display-mode: "form" }
%cd /content
!rm -rf PULSIM
!git clone --quiet --branch main https://github.com/phillip-jaeslee/PULSIM
%cd PULSIM
!pip install --quiet ".[viz]"
# Rerunning this cell picks up the latest main: forget any PULSIM already
# imported in this session, so the next import loads the fresh install.
import sys
for _m in [m for m in sys.modules if m == "PULSIM" or m.startswith("PULSIM.")]:
    del sys.modules[_m]

import os
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import ipywidgets as widgets
from IPython.display import display, clear_output

from PULSIM import RFShape, Pulse, PulseSequence, NumpyBackend
from PULSIM.spin_system import SpinSystem, gyro_ratio
from PULSIM.liouville import Delay, ShapePulseSegment, LiouvilleSequence
from PULSIM.sequence_figure import draw_sequence
from PULSIM.gradients import uniform_positions, gradient_offsets

# ---- figure style: the same rules as PULSIM_colab.ipynb ---------------------
# 7 in wide (3.5 in for a figure that is one profile alone), all text 8 pt, a
# figure's own title 10 pt; one curve is black, colors only tell curves apart.
mpl.rcParams["font.family"] = "Liberation Sans"
FIG_W, FIG_W_COL = 7.0, 3.5
mpl.rcParams.update({"font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "figure.titlesize": 10})
M_COMPONENTS = {"Mx": "#1f77b4", "My": "#ff7f0e", "Mz": "#2ca02c", "|Mxy|": "#000000"}
PALETTE = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]


# ---- nucleus labels: the isotope as a superscript, H -> 1H (as in PULSIM_colab) --
_ISO = {"H": "1H", "D": "2H", "T": "3H"}
_SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")


def nuc_label(ch):
    """For printed text and labels: 'H' -> '¹H', '13C' -> '¹³C'."""
    import re
    m = re.fullmatch(r"(\d+)([A-Za-z]+)", _ISO.get(ch, ch))
    return m.group(1).translate(_SUP) + m.group(2) if m else ch


# ---- units --------------------------------------------------------------------
def ppm_to_khz(ppm, ref_ppm, mhz):
    """Offset (kHz) of a shift `ppm` from `ref_ppm`, for a nucleus observed at mhz."""
    return (np.asarray(ppm, dtype=float) - ref_ppm) * mhz / 1000.0


def nucleus_mhz(b0_mhz, nucleus):
    """Frequency (MHz) of `nucleus` in the magnet whose 1H frequency is b0_mhz."""
    return b0_mhz * abs(gyro_ratio(nucleus) / gyro_ratio("H"))


def parse_values(text):
    """'80, 100, 120' -> [80.0, 100.0, 120.0]"""
    return [float(v) for v in text.replace(";", ",").split(",") if v.strip()]


# ---- pulses and magnetization -------------------------------------------------
PHASE_AXIS = {"x": "x", "y": "y", "-x": np.pi, "-y": -np.pi / 2}


def hard_pulse(flip_deg, phase, nu1_khz, nucleus="H", points=10):
    """A rectangular pulse of flip_deg at RF strength nu1_khz (kHz): it lasts
    flip / (360 nu1) ms."""
    shape = RFShape.create("hard", duration=abs(flip_deg) / 360.0 / nu1_khz, points=points)
    return Pulse(shape, flip=np.deg2rad(abs(flip_deg)), axis=PHASE_AXIS[phase],
                 backend=NumpyBackend(Gamma=gyro_ratio(nucleus)))


class OffsetPulse(Pulse):
    """A Pulse played offset_khz away from the carrier. Its RF carries the phase
    ramp exp(+i 2 pi nu t), zero at `align` of the pulse (0 start, 0.5 center,
    1 end -- the meaning of Bruker's SPOAL). The amplitude is specified as for
    the same pulse on resonance; only the phase changes. (The package's Pulse
    has no offset of its own; the sign was checked: the pulse acts at
    +offset_khz, for positive and negative gamma alike.)"""

    def __init__(self, shape, flip=None, axis="x", backend=None, offset_khz=0.0, align=0.5):
        super().__init__(shape, flip, axis=axis, backend=backend)
        self.offset_khz, self.align = float(offset_khz), float(align)

    def calibrated_rf(self):
        rf = super().calibrated_rf()
        t = (np.arange(len(rf)) + 0.5) * self.shape.dt
        return rf * np.exp(2j * np.pi * self.offset_khz * (t - self.align * self.shape.duration))


def magnetization(n, along="z"):
    """(3, n) unit magnetization along x, y or z."""
    M = np.zeros((3, n))
    M["xyz".index(along)] = 1.0
    return M


def component(M, name):
    """Mx, My, Mz or |Mxy| from M of shape (3, n)."""
    return np.hypot(M[0], M[1]) if name == "|Mxy|" else M["xyz".index(name[-1])]


def separator():
    """A thin line between groups of controls (as in PULSIM_colab.ipynb)."""
    return widgets.HTML('<hr style="border:0;border-top:1px solid #d0d0d0;margin:6px 0">',
                        layout=widgets.Layout(width="100%"))


def tidy_labels(ax):
    """Short pulse labels: a hard pulse is drawn as a rectangle anyway, so
    'hard 21°x' becomes '21°x' -- trains of short pulses drawn to scale then
    keep their labels apart."""
    for t in ax.texts:
        if t.get_text().startswith("hard "):
            t.set_text(t.get_text()[5:].lstrip())


def show_segments(segs, ss, title="the sequence as simulated", to_scale=True, gradients=()):
    """Draw density-matrix segments (Delay, IdealPulse, ShapePulseSegment, ...)
    with draw_sequence, plus gradients on a G line under the RF lines (the
    package's draw_sequence has none). `gradients`: positions in `segs` of the
    Delays that are gradient pulses, each as an index or as
    (index, relative strength -1..1, "sine" | "rectangular")."""
    lseq = LiouvilleSequence(segs, ss)
    n = ss.n_spins
    fig, ax = plt.subplots(figsize=(FIG_W, 0.9 + 0.75 * n + (0.6 if gradients else 0.3)))
    draw_sequence(lseq, ax=ax, to_scale=to_scale)
    tidy_labels(ax)
    if gradients:
        # the slots draw_sequence gave each element (same layout code); delays are
        # never merged in the drawing, so the k-th Delay segment is the k-th delay slot
        from PULSIM import sequence_figure as sf
        items = sf._attach_to_shaped(sf._group_simultaneous(sf.describe_sequence(lseq)))
        positions, total, _ = sf._layout(items, to_scale, 0.012)
        delay_slots = [pos for it, pos in zip(items, positions) if it["kind"] == "delay"]
        delay_segs = [i for i, sg in enumerate(segs) if isinstance(sg, Delay)]
        if len(delay_slots) != len(delay_segs):
            raise RuntimeError("cannot place the gradients: the drawing merged delays")
        y = -1.3                                       # under the delay arrows
        ax.plot([0, total], [y, y], color="0.25", lw=1.0, zorder=1)
        ax.text(-0.015 * total, y, "G", ha="right", va="center", fontsize=11)
        for gr in gradients:
            i, rel, kind = (gr, 1.0, "sine") if np.isscalar(gr) else gr
            x0, w = delay_slots[delay_segs.index(i)]
            # the whole slot; height at full strength as a shaped pulse's peak (0.26)
            xs = np.linspace(x0 + 0.03 * w, x0 + 0.97 * w, 60)
            u = (xs - xs[0]) / (xs[-1] - xs[0])
            prof = np.sin(np.pi * u) if kind == "sine" else np.where((u > 0) & (u < 1), 1.0, 0.0)
            ax.fill_between(xs, y, y + 0.26 * rel * prof, color="0.6", lw=0.8,
                            edgecolor="0.25", zorder=2)
        lo, hi = ax.get_ylim()
        ax.set_ylim(min(lo, y - 0.35), hi)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def show_sequence(seq, nucleus="H", title="the sequence as simulated", to_scale=True,
                  gradients=()):
    """A Bloch PulseSequence (Pulses and Delays on one nucleus), drawn by
    show_segments; `gradients` as there."""
    ss = SpinSystem(nuclei=[nucleus], offsets=[0.0])
    segs = [p if isinstance(p, Delay) else ShapePulseSegment(p) for p in seq]
    show_segments(segs, ss, title, to_scale, gradients)


# ---- saving: downloads, figures (as in PULSIM_colab.ipynb) and data ------------
def _stamp(stem):
    return f"{stem}_{time.strftime('%Y%m%d_%H%M%S')}"


_PENDING_DOWNLOADS = []


def offer_file(path):
    """In Colab, queue the file; the run's single download happens in
    flush_downloads(). Elsewhere, show a link to it right away."""
    size = os.path.getsize(path) / 1e6
    try:
        import google.colab  # noqa: F401
    except ImportError:
        from IPython.display import FileLink
        print(f"  saved {path} ({size:.1f} MB) in the working directory:")
        display(FileLink(path))
        return
    _PENDING_DOWNLOADS.append(path)
    print(f"  saved {path} ({size:.1f} MB) -- downloads when the run finishes "
          f"(also in the Colab Files panel)")


def flush_downloads():
    """Call LAST in a run. Colab's files.download works by putting JavaScript in
    the output area; any output added after it can make Colab redraw the area and
    run that script again -- the same file then downloads twice. So: one download
    call per run, with nothing printed after it. Several files from one run go
    out together as one .zip."""
    if not _PENDING_DOWNLOADS:
        return
    paths = list(_PENDING_DOWNLOADS)
    _PENDING_DOWNLOADS.clear()
    if len(paths) == 1:
        target = paths[0]
    else:
        import zipfile
        target = _stamp("pulsim_run") + ".zip"
        with zipfile.ZipFile(target, "w") as z:
            for p in paths:
                z.write(p)
        print(f"  {len(paths)} files from this run bundled into {target}")
    from google.colab import files
    files.download(target)                     # the very last output of the run


# ---- saving the static figures of a run: PNG / PDF / SVG ---------------------
FIG_FORMATS = ["no", "PNG (300 dpi)", "PDF", "SVG"]


class save_figures:
    """Every figure shown (plt.show) inside this block is also saved, then the
    run's figures go out as ONE file: a multi-page PDF, or a .zip of PNG/SVG.

        with save_figures(w_figsave.value, "pulsim_single"):
            ...make and show figures as usual...
    """

    def __init__(self, fmt="no", stem="pulsim_figures"):
        self.fmt, self.stem, self.paths, self.pdf = fmt, stem, [], None

    def __enter__(self):
        self._show = plt.show
        if self.fmt == "no":
            return self
        self.base = _stamp(self.stem)
        if self.fmt == "PDF":
            from matplotlib.backends.backend_pdf import PdfPages
            self.pdf = PdfPages(self.base + ".pdf")

        def show(*args, **kwargs):
            fig = plt.gcf()
            if fig.axes:                                  # skip empty figures
                if self.pdf is not None:
                    self.pdf.savefig(fig, bbox_inches="tight")
                else:
                    ext = "png" if self.fmt.startswith("PNG") else "svg"
                    title = next((a.get_title() for a in fig.axes if a.get_title()), "figure")
                    slug = "".join(c if c.isalnum() else "_" for c in title).strip("_")[:40]
                    path = f"{self.base}_{len(self.paths) + 1}_{slug}.{ext}"
                    fig.savefig(path, dpi=300, bbox_inches="tight")
                    self.paths.append(path)
            return self._show(*args, **kwargs)

        plt.show = show
        return self

    def __exit__(self, exc_type, exc, tb):
        plt.show = self._show                             # always restore
        if self.fmt == "no":
            return False
        if exc_type is not None:                          # the run failed: no half bundle
            if self.pdf is not None:
                self.pdf.close()
                os.remove(self.base + ".pdf")
            for p in self.paths:
                os.remove(p)
            return False
        if self.pdf is not None:
            n = self.pdf.get_pagecount()
            self.pdf.close()
            if n:
                print(f"  figures: {n} page(s) in one PDF")
                offer_file(self.base + ".pdf")
            else:
                os.remove(self.base + ".pdf")
        elif self.paths:
            import zipfile
            path = self.base + ".zip"
            with zipfile.ZipFile(path, "w") as z:
                for p in self.paths:
                    z.write(p)
                    os.remove(p)
            print(f"  figures: {len(self.paths)} {self.fmt.split()[0]} file(s) in one .zip")
            offer_file(path)
        return False                                       # never swallow an error

DATA_FORMATS = ["no", "CSV"]


def save_csv(stem, columns):
    """Write {column name: 1-D array} (equal lengths) as one CSV and offer it."""
    import csv
    path = _stamp(stem) + ".csv"
    names = list(columns)
    with open(path, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(names)
        for row in zip(*(np.asarray(columns[k], dtype=float) for k in names)):
            w.writerow([f"{v:.8g}" for v in row])
    print(f"  data: {len(names)} columns x {len(next(iter(columns.values())))} rows")
    offer_file(path)


print("PULSIM ready.")

In [ ]:
#@title 1. Water suppression { display-mode: "form" }
def water_widget():
    from math import comb
    SHAPES = sorted(s for s in RFShape.available() if s not in ("hard", "file", "composite", "varian"))
    GRAD_SHAPES = ["sine", "rectangular"]
    GRAD_AREA = {"sine": 2.0 / np.pi, "rectangular": 1.0}      # mean amplitude / peak
    SEG_KINDS = ["hard pulse", "delay", "shaped pulse", "gradient"]

    # ---- presets: rows (kind, flip, phase, dur ms, shape, points, TR, G %) ------------
    def binomial(weights, phases, alpha, d, nu1=25.0):
        """Hard pulses of weight * alpha degrees at nu1 kHz, separated by delays d (ms)."""
        rows = []
        for k, (w, ph) in enumerate(zip(weights, phases)):
            rows.append(("hard pulse", w * alpha, ph, (w * alpha / 360.0) / nu1))
            if k < len(weights) - 1:
                rows.append(("delay", 0.0, "x", d))
        return rows

    def binomial_n(n, total=90.0, d=0.224, nu1=25.0):
        """Binomial train of order n: n + 1 pulses with weights from Pascal's
        triangle and alternating phases (x, -x, ...); the unit angle is the total
        flip divided by the sum of the weights, 2**n."""
        w = [comb(n, k) for k in range(n + 1)]
        return binomial(w, ["x" if k % 2 == 0 else "-x" for k in range(n + 1)],
                        total / sum(w), d, nu1)

    def grad(dur=1.0, g=40.0, shape="sine"):
        return ("gradient", 0.0, "x", dur, shape, 10, False, g)

    PRESETS = {
        "Jump-return 1-1": dict(
            rows=binomial([1, 1], ["x", "-x"], 90.0, 0.101), show=("Mx",),
            note="90x – d – 90−x (Plateau & Guéron 1982). Water stays on z; the amides come "
                 "out with +Mx and the aliphatics with −Mx. Maximum near 1/(4d) from the "
                 "pulses, next null near 1/(2d)."),
        "Binomial (any order)": dict(
            rows=binomial_n(3), show=("|Mxy|",), generator=True,
            note="Pascal's-triangle weights with alternating phases (Hore 1983); set the "
                 "order and the total flip below and the rows and angles follow. Order 3 "
                 "is 1-3-3-1: a wider, flatter water null than jump-return and a narrower "
                 "excitation band. Maximum near 1/(2d), next null near 1/d."),
        "WATERGATE 3-9-19": dict(
            rows=([("hard pulse", 90.0, "x", 0.01), grad(), ("delay", 0.0, "x", 0.2)]
                  + binomial([3, 9, 19, 19, 9, 3], ["x", "x", "x", "-x", "-x", "-x"],
                             180.0 / 26, 0.200)
                  + [grad(), ("delay", 0.0, "x", 0.2)]),
            show=("|Mxy|",),
            note="90° – G – 3α-9α-19α-19α̅-9α̅-3α̅ (26α = 180°) – G (Piotto et al. 1992; "
                 "Sklenář et al. 1993): the amides are refocused between the two gradients "
                 "and come back; water is left alone by the element and dephased. Make the "
                 "two gradients unequal to see the echo get lost."),
        "Build your own": dict(
            rows=[("hard pulse", 90.0, "x", 0.01)], show=("Mx", "My", "Mz"),
            note="Start from one hard pulse: add segments with + Add segment, remove rows "
                 "with ✕."),
    }
    SWEEPS = {"none": None, "all delays (µs)": "d", "hard pulses ν1 (kHz)": "nu1",
              "pulses at (ppm)": "carrier", "B0 (MHz)": "b0", "last gradient (%)": "g_last"}
    DEFAULTS = dict(b0=600.0, water=4.7, carrier=4.7, hi=11.0, lo=-1.0, n=601,
                    gmax=50.0, sample=1.5, sweep="none", values="", figsave="no", data="no")

    S = {"description_width": "initial"}
    def W(px): return widgets.Layout(width=px)
    def L(text, px): return widgets.Label(text, layout=W(px))

    w_preset  = widgets.Dropdown(options=list(PRESETS), description="preset:", style=S,
                                 layout=W("230px"))
    preset_note = widgets.HTML()
    w_b0      = widgets.FloatText(description="B0 (¹H MHz):", style=S, layout=W("160px"))
    w_water   = widgets.FloatText(description="water (ppm):", style=S, layout=W("150px"))
    w_carrier = widgets.FloatText(description="pulses at (ppm):", style=S, layout=W("170px"),
                                  tooltip="the frequency of the pulses: their own offset")
    w_hi      = widgets.FloatText(description="from (ppm):", style=S, layout=W("140px"))
    w_lo      = widgets.FloatText(description="to (ppm):", style=S, layout=W("125px"))
    w_n       = widgets.IntText(description="points:", style=S, layout=W("120px"))
    w_gmax    = widgets.FloatText(description="G max (G/cm):", style=S, layout=W("160px"),
                                  tooltip="the gradient at 100 %")
    w_sample  = widgets.FloatText(description="sample length (cm):", style=S, layout=W("190px"),
                                  tooltip="length of the sample along the gradient (the part "
                                          "the coil sees), centered on the gradient")
    w_show    = {c: widgets.Checkbox(value=False, description=c, indent=False, layout=W("68px"))
                 for c in M_COMPONENTS}
    w_sweep   = widgets.Dropdown(options=list(SWEEPS), description="compare:", style=S,
                                 layout=W("230px"))
    w_values  = widgets.Text(description="values:", style=S, layout=W("220px"),
                             placeholder="e.g. 80, 101, 120")
    w_figsave = widgets.Dropdown(options=FIG_FORMATS, description="figures:", style=S,
                                 layout=W("190px"))
    w_data    = widgets.Dropdown(options=DATA_FORMATS, description="data:", style=S,
                                 layout=W("140px"))
    w_order   = widgets.BoundedIntText(value=3, min=1, max=10, description="order n:", style=S,
                                       layout=W("130px"))
    w_total   = widgets.FloatText(value=90.0, description="total flip (°):", style=S,
                                  layout=W("160px"))
    w_gen_d   = widgets.FloatText(value=224.0, description="d (µs):", style=S, layout=W("130px"))
    w_gen_nu1 = widgets.FloatText(value=25.0, description="ν1 (kHz):", style=S, layout=W("140px"))
    gen_info  = widgets.HTML()
    gen_box   = widgets.VBox([widgets.HBox([w_order, w_total, w_gen_d, w_gen_nu1]), gen_info])
    out = widgets.Output()

    # ---- one row per segment (the main notebook's Bloch rows, plus gradients) ----------
    _rows = []
    seg_box = widgets.VBox([])

    def _refresh():
        for n, r in enumerate(_rows, start=1):
            r["idx"].value = str(n)
            r["up"].disabled = (n == 1)
            r["down"].disabled = (n == len(_rows))
        seg_box.children = tuple(r["box"] for r in _rows)

    def _move(row, step):
        i = _rows.index(row)
        j = i + step
        if 0 <= j < len(_rows):
            _rows[i], _rows[j] = _rows[j], _rows[i]
            _refresh()

    def _make_row(kind="hard pulse", flip=90.0, phase="x", dur=0.01, shape="gausscasq5",
                  pts=10, tr=False, g=40.0):
        idx  = L("", "26px")
        w_kd = widgets.Dropdown(options=SEG_KINDS, value=kind, layout=W("120px"))
        w_fl = widgets.FloatText(value=round(flip, 6), layout=W("75px"))
        w_ph = widgets.Dropdown(options=["x", "y", "-x", "-y"], value=phase, layout=W("60px"))
        w_du = widgets.FloatText(value=round(dur, 6), layout=W("80px"))
        opts = GRAD_SHAPES if kind == "gradient" else SHAPES
        w_sh = widgets.Dropdown(options=opts, value=shape if shape in opts else opts[0],
                                layout=W("130px"))
        w_pt = widgets.IntText(value=pts, layout=W("75px"))
        w_tr = widgets.Checkbox(value=tr, indent=False, layout=W("40px"),
                                tooltip="time-reversed: play this shape backwards")
        w_g  = widgets.FloatText(value=g, layout=W("70px"),
                                 tooltip="gradient strength, % of G max (negative: reversed)")
        w_up = widgets.Button(description="↑", tooltip="move this row up", layout=W("32px"))
        w_dn = widgets.Button(description="↓", tooltip="move this row down", layout=W("32px"))
        w_rm = widgets.Button(description="✕", tooltip="remove this row", layout=W("36px"))

        def _sync(change=None):
            k = w_kd.value
            want = GRAD_SHAPES if k == "gradient" else SHAPES
            if tuple(w_sh.options) != tuple(want):     # the shape column changes meaning
                w_sh.options = want
                w_sh.value = "sine" if k == "gradient" else "gausscasq5"
            w_fl.disabled = w_ph.disabled = k in ("delay", "gradient")
            w_pt.disabled = k in ("delay", "gradient")
            w_sh.disabled = k not in ("shaped pulse", "gradient")
            w_tr.disabled = (k != "shaped pulse")
            w_g.disabled = (k != "gradient")
        w_kd.observe(_sync, names="value")
        _sync()

        def _remove(_b):
            if len(_rows) == 1:
                with out:
                    clear_output(wait=True)
                    print("A sequence needs at least one row.")
                return
            _rows.remove(row)
            _refresh()
        w_rm.on_click(_remove)
        w_up.on_click(lambda _b: _move(row, -1))
        w_dn.on_click(lambda _b: _move(row, +1))
        row = {"idx": idx, "kind": w_kd, "flip": w_fl, "phase": w_ph, "dur": w_du,
               "shape": w_sh, "pts": w_pt, "tr": w_tr, "g": w_g, "up": w_up, "down": w_dn,
               "box": widgets.HBox([idx, w_kd, w_fl, w_ph, w_du, w_sh, w_pt, w_tr, w_g,
                                    w_up, w_dn, w_rm])}
        return row

    def _add(_b=None):
        if _rows:                                   # a new row copies the last one
            p = _rows[-1]
            _rows.append(_make_row(*(p[k].value for k in
                                     ("kind", "flip", "phase", "dur", "shape", "pts", "tr", "g"))))
        else:
            _rows.append(_make_row())
        _refresh()

    def _generate(change=None):
        """Binomial preset: rebuild the rows from the order and the total flip."""
        if not PRESETS[w_preset.value].get("generator"):
            return
        n = int(w_order.value)
        if w_gen_d.value <= 0 or w_gen_nu1.value <= 0:
            gen_info.value = "<i>d and ν1 must be positive</i>"
            return
        w = [comb(n, k) for k in range(n + 1)]
        alpha = w_total.value / sum(w)
        _rows.clear()
        for a in binomial_n(n, w_total.value, w_gen_d.value / 1000.0, w_gen_nu1.value):
            _rows.append(_make_row(*a))
        _refresh()
        gen_info.value = (f"<i>{'-'.join(map(str, w))}, phases "
                          f"{', '.join('x' if k % 2 == 0 else '−x' for k in range(n + 1))}; "
                          f"α = {w_total.value:g}° / {sum(w)} = {alpha:.4g}°; maximum near "
                          f"1/(2d) = {500.0 / w_gen_d.value:.3g} kHz from the pulses</i>")

    def _load_preset(change=None):
        p, d = PRESETS[w_preset.value], DEFAULTS
        _rows.clear()
        for a in p["rows"]:
            _rows.append(_make_row(*a))
        _refresh()
        for c, w in w_show.items():
            w.value = c in p["show"]
        w_b0.value, w_water.value, w_carrier.value = d["b0"], d["water"], d["carrier"]
        w_hi.value, w_lo.value, w_n.value = d["hi"], d["lo"], d["n"]
        w_gmax.value, w_sample.value = d["gmax"], d["sample"]
        w_sweep.value, w_values.value = d["sweep"], d["values"]
        w_figsave.value, w_data.value = d["figsave"], d["data"]
        preset_note.value = f"<i>{p['note']}</i>"
        gen_box.layout.display = "" if p.get("generator") else "none"
        if p.get("generator"):                     # back to the preset's own order and angles
            w_order.value, w_total.value, w_gen_d.value, w_gen_nu1.value = 3, 90.0, 224.0, 25.0
            _generate()
        if change is not None:
            with out:
                clear_output(wait=True)
                print(f"Loaded preset: {w_preset.value}")

    w_preset.observe(_load_preset, names="value")
    for _w in (w_order, w_total, w_gen_d, w_gen_nu1):
        _w.observe(_generate, names="value")

    # ---- the experiment ---------------------------------------------------------
    def _params():
        rows = [dict(kind=r["kind"].value, flip=r["flip"].value, phase=r["phase"].value,
                     dur=r["dur"].value, shape=r["shape"].value, pts=int(r["pts"].value),
                     tr=r["tr"].value, g=r["g"].value) for r in _rows]
        return dict(b0=w_b0.value, water=w_water.value, carrier=w_carrier.value,
                    gmax=w_gmax.value, sample=w_sample.value, rows=rows,
                    d=None, nu1=None, g_last=None)

    def _rows_for(p):
        """The rows with a compare value applied (last gradient)."""
        rows = [dict(r) for r in p["rows"]]
        if p["g_last"] is not None:
            last = [i for i, r in enumerate(rows) if r["kind"] == "gradient"]
            if not last:
                raise ValueError("there is no gradient row to compare")
            rows[last[-1]]["g"] = p["g_last"]
        return rows

    def _element(r, p):
        if r["kind"] in ("delay", "gradient"):
            dur = (p["d"] / 1000.0) if (p["d"] is not None and r["kind"] == "delay") else r["dur"]
            return Delay(dur)
        if r["kind"] == "hard pulse":
            dur = (r["flip"] / 360.0 / p["nu1"]) if p["nu1"] is not None else r["dur"]
            shape = RFShape.create("hard", duration=dur, points=max(2, r["pts"]))
        else:
            shape = RFShape.create(r["shape"], duration=r["dur"], points=r["pts"],
                                   time_reversed=bool(r["tr"]))
        if shape.calibration_mode == "adiabatic":
            return Pulse(shape, axis=PHASE_AXIS[r["phase"]], backend=NumpyBackend(Gamma=gyro_ratio("H")))
        return Pulse(shape, flip=np.deg2rad(r["flip"]), axis=PHASE_AXIS[r["phase"]],
                     backend=NumpyBackend(Gamma=gyro_ratio("H")))

    def _turns(r, p):
        """Phase difference (turns) a gradient row puts across the sample."""
        G = r["g"] / 100.0 * p["gmax"] * 10.0 * GRAD_AREA[r["shape"]]   # mean mT/m (1 G/cm = 10 mT/m)
        return gyro_ratio("H") * G * (p["sample"] / 100.0) * r["dur"]    # kHz * ms

    # Cartesian (Mx, My, Mz) <-> spherical (M+, Mz, M-): a gradient multiplies
    # M+ and M- by opposite phases and leaves Mz alone.
    TO_SPH = np.array([[1, 1j, 0], [0, 0, 1], [1, -1j, 0]])
    TO_CART = np.linalg.inv(TO_SPH)

    def _simulate(p, ppm):
        """Magnetization (3, n) at every shift in ppm, read in the frame of the pulses.

        Gradients: between two gradients the pulses and delays act on each
        offset as one rotation, found once from Mx, My and Mz. A gradient adds a
        phase that changes linearly across the sample: it splits the
        magnetization into coherence pathways (M+, Mz, M-), each carrying the
        phase difference K (turns) it has gathered across the sample. The
        receiver averages over a uniform sample, which multiplies each pathway
        by sin(pi K) / (pi K) -- exactly, for a continuous sample, with no
        slices. Chemical shift evolves during a gradient as during a delay."""
        rows = _rows_for(p)
        df = ppm_to_khz(ppm, p["carrier"], p["b0"])          # offset from the pulses, kHz
        n = len(df)
        if not any(r["kind"] == "gradient" for r in rows):
            return PulseSequence([_element(r, p) for r in rows]).run(magnetization(n, "z"), df)

        def rotation(chunk):
            """(n, 3, 3) spherical rotation of a run of pulses and delays."""
            if not chunk:
                return None
            seq = PulseSequence(chunk)
            R = np.stack([seq.run(magnetization(n, a), df) for a in "xyz"], axis=2)   # (3, n, 3)
            R = np.transpose(R, (1, 0, 2))                                            # (n, 3, 3)
            return TO_SPH @ R @ TO_CART

        paths = {0.0: np.einsum("ij,jn->ni", TO_SPH, magnetization(n, "z"))}    # K -> (n, 3)
        chunk = []
        def apply(chunk):
            R = rotation(chunk)
            if R is not None:
                for K in paths:
                    paths[K] = np.einsum("nij,nj->ni", R, paths[K])
            chunk.clear()
        for r in rows:
            if r["kind"] != "gradient":
                chunk.append(_element(r, p))
                continue
            chunk.append(Delay(r["dur"]))                # the chemical shift during it
            apply(chunk)
            T = _turns(r, p)
            new = {}
            for K, v in paths.items():
                for comp, sgn in ((0, +1), (1, 0), (2, -1)):    # M+, Mz, M-
                    w = np.zeros_like(v)
                    w[:, comp] = v[:, comp]
                    k2 = round(K + sgn * T, 9)
                    new[k2] = new[k2] + w if k2 in new else w
            paths.clear()
            paths.update({K: v for K, v in new.items() if np.abs(v).max() > 1e-14})
        apply(chunk)
        total = sum(v * np.sinc(K) for K, v in paths.items())       # np.sinc(x) = sin(pi x)/(pi x)
        return np.real(np.einsum("ij,nj->in", TO_CART, total))

    # ---- figures ----------------------------------------------------------------
    def _decorate(ax, p, top=True):
        ax.set_xlim(w_hi.value, w_lo.value)                        # NMR convention
        ax.axhline(0, color="0.8", lw=0.5)
        ax.axvline(p["water"], color="0.4", ls=":", lw=0.9)
        ax.set(ylim=(-1.05, 1.05), yticks=[-1, -0.5, 0, 0.5, 1])
        if abs(p["carrier"] - p["water"]) > 1e-9:
            ax.axvline(p["carrier"], color="0.4", ls="--", lw=0.9)
        if top:                                   # labels inside, under the top axis
            tr = ax.get_xaxis_transform()
            ax.text(p["water"], 0.97, " water", transform=tr, ha="left", va="top", color="0.3")
            if abs(p["carrier"] - p["water"]) > 1e-9:
                ax.text(p["carrier"], 0.88, " pulses", transform=tr, ha="left", va="top",
                        color="0.3")

    def _top_axis(ax, p):
        f = lambda x: (x - p["carrier"]) * p["b0"] / 1000.0
        g = lambda k: k * 1000.0 / p["b0"] + p["carrier"]
        ax.secondary_xaxis("top", functions=(f, g)).set_xlabel("offset from the pulses (kHz)")

    def _report(label, p, ppm, M):
        i_w = int(np.argmin(np.abs(ppm - p["water"])))
        mxy = np.hypot(M[0], M[1])
        side = {}
        for name, mask in (("downfield", ppm > p["water"]), ("upfield", ppm < p["water"])):
            if mask.any():
                j = np.flatnonzero(mask)[np.argmax(mxy[mask])]
                side[name] = f"{mxy[j]:.3f} at {ppm[j]:.2f} ppm"
        print(f"  {label}: at water Mx {M[0, i_w]:+.3f}, My {M[1, i_w]:+.3f}, "
              f"Mz {M[2, i_w]:+.3f}, |Mxy| {mxy[i_w]:.4f};  largest |Mxy| "
              + "; ".join(f"{k} {v}" for k, v in side.items()))

    def _run(_b):
        with out:
            clear_output(wait=True)
            try:
                print("Run button has been clicked. Now running... Please wait...")
                shown = [c for c, w in w_show.items() if w.value]
                if not shown:
                    raise ValueError("tick at least one of Mx, My, Mz, |Mxy|")
                base = _params()
                key = SWEEPS[w_sweep.value]
                values = parse_values(w_values.value) if key else []
                if key and not values:
                    raise ValueError("type the values to compare, e.g. 80, 101, 120")
                name, unit = (w_sweep.value[:-1].split(" (") if key else ("", ""))
                runs = ([(f"{name} = {v:g} {unit}", {**base, key: v}) for v in values]
                        if key else [("", base)])
                ppm = np.linspace(w_hi.value, w_lo.value, int(w_n.value))

                rows = base["rows"]
                pul = [r for r in rows if r["kind"] in ("hard pulse", "shaped pulse")]
                dls = [r["dur"] * 1000 for r in rows if r["kind"] == "delay"]
                print(f"  {len(pul)} pulse(s) of " + ", ".join(f"{r['dur'] * 1000:.1f}" for r in pul)
                      + " µs" + (", delays " + ", ".join(f"{t:g}" for t in dls) + " µs" if dls else ""))
                if dls:
                    d = dls[0]
                    print(f"  first delay d = {d:g} µs: 1/(4d) = {250 / d:.2f} kHz, "
                          f"1/(2d) = {500 / d:.2f} kHz from the pulses")
                for r in rows:
                    if r["kind"] == "gradient":
                        print(f"  gradient {r['g']:g} % of {base['gmax']:g} G/cm, {r['shape']}, "
                              f"{r['dur']:g} ms: {_turns(r, base):+.1f} turns across "
                              f"{base['sample']:g} cm")
                t0 = time.time()
                results = [(label, p, _simulate(p, ppm)) for label, p in runs]
                print(f"  {len(results)} run(s) x {len(ppm)} shifts in {time.time() - t0:.2f} s")
                for label, p, M in results:
                    _report(label or "result", p, ppm, M)

                stem = "pulsim_water_" + w_preset.value.split()[0].lower()
                name = w_preset.value
                if PRESETS[name].get("generator"):          # name the train actually run
                    n = int(w_order.value)
                    name = "Binomial " + "-".join(str(comb(n, k)) for k in range(n + 1))
                title = f"{name}, {base['b0']:g} MHz"
                with save_figures(w_figsave.value, stem):
                    seq = [_element(r, base) for r in rows]
                    gmax_abs = max([abs(r["g"]) for r in rows if r["kind"] == "gradient"] or [1.0]) or 1.0
                    grads = [(i, r["g"] / gmax_abs, r["shape"]) for i, r in enumerate(rows)
                             if r["kind"] == "gradient"]
                    show_sequence(seq, to_scale=True, gradients=grads,
                                  title="the sequence as simulated")
                    if not key:                       # one profile: one column wide
                        fig, ax = plt.subplots(figsize=(FIG_W_COL, 3.0))
                        M = results[0][2]
                        for c in shown:
                            ax.plot(ppm, component(M, c), lw=1.4, label=c,
                                    color="k" if len(shown) == 1 else M_COMPONENTS[c])
                        _decorate(ax, base)
                        _top_axis(ax, base)
                        ax.set(xlabel="¹H chemical shift (ppm)", ylabel="magnetization")
                        ax.legend(loc="lower left", frameon=False)
                        fig.suptitle(title)
                    else:                             # one panel per component, one curve per value
                        fig, axs = plt.subplots(len(shown), 1, sharex=True, squeeze=False,
                                                figsize=(FIG_W, 0.9 + max(2.4, 1.2 * len(shown))))
                        axs = axs[:, 0]
                        for a, c in zip(axs, shown):
                            for k, (label, p, M) in enumerate(results):
                                a.plot(ppm, component(M, c), lw=1.3,
                                       color=PALETTE[k % len(PALETTE)], label=label)
                            _decorate(a, base, top=a is axs[0])
                            a.set_ylabel(c)
                        if key not in ("carrier", "b0"):       # one pulse frequency for all runs
                            _top_axis(axs[0], base)
                        axs[-1].set_xlabel("¹H chemical shift (ppm)")
                        axs[0].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
                        fig.suptitle(title)
                    fig.tight_layout()
                    plt.show()
                if w_data.value == "CSV":
                    cols = {"ppm": ppm}
                    if key not in ("carrier", "b0"):
                        cols["offset from the pulses (kHz)"] = ppm_to_khz(ppm, base["carrier"],
                                                                          base["b0"])
                    for label, p, M in results:
                        for c in M_COMPONENTS:
                            cols[f"{c} [{label}]" if label else c] = component(M, c)
                    save_csv(stem, cols)
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")
            finally:
                flush_downloads()      # last: nothing may print after a Colab download

    b_add   = widgets.Button(description="+ Add segment", layout=W("130px"))
    b_run   = widgets.Button(description="Run", button_style="primary", layout=W("110px"))
    b_reset = widgets.Button(description="Reload preset", layout=W("120px"),
                             tooltip="restore the selected preset (undo your edits)")
    b_add.on_click(_add)
    b_run.on_click(_run)
    b_reset.on_click(lambda _b: _load_preset(_b))
    _load_preset()

    header = widgets.HBox([L("#", "26px"), L("segment", "120px"), L("flip °", "75px"),
                           L("phase", "60px"), L("dur ms", "80px"), L("shape", "130px"),
                           L("points", "75px"), L("TR", "40px"), L("G %", "70px")])
    display(widgets.VBox([
        widgets.HBox([w_preset, preset_note]),
        separator(),
        widgets.HTML("<b>Segments</b> — applied in order to one ¹H; a hard pulse's strength "
                     "is its flip / duration; TR = time-reversed shaped pulse; a gradient "
                     "row takes a shape (sine or rectangular) and G % of G max"),
        gen_box,
        header, seg_box,
        separator(),
        widgets.HTML("<b>Spectrometer</b> — shifts in ppm"),
        widgets.HBox([w_b0, w_water, w_carrier]),
        widgets.HBox([w_gmax, w_sample]),
        widgets.HBox([w_hi, w_lo, w_n]),
        separator(),
        widgets.HTML("<b>Figure</b>"),
        widgets.HBox([L("show:", "40px")] + list(w_show.values())),
        widgets.HBox([w_sweep, w_values]),
        separator(),
        widgets.HBox([b_add, b_run, b_reset, w_figsave, w_data]),
        out,
    ]))


water_widget()

In [ ]:
#@title 2. Bloch–Siegert shift { display-mode: "form" }
def bloch_siegert_widget():
    SHAPES = sorted(s for s in RFShape.available() if s not in ("hard", "file", "composite", "varian"))
    SEG_KINDS = ["hard pulse", "delay", "shaped pulse"]
    ALIGNS = [("start", 0.0), ("center", 0.5), ("end", 1.0)]
    FIGURES = ["Phase vs shift", "Compensation"]
    SWEEPS = {"none": None, "first pulse: duration (ms)": "dur", "first pulse: at (ppm)": "at",
              "first pulse: flip (°)": "flip", "B0 (MHz)": "b0"}
    NUCLEI = [("¹H", "H"), ("¹³C", "13C"), ("¹⁵N", "15N")]

    # ---- presets: rows (kind, flip, phase, dur ms, shape, points, TR, at ppm, ramp zero)
    def q3(at):
        return ("shaped pulse", 180.0, "x", 0.35, "gausscasq3", 500, False, at, 0.5)
    C13 = dict(nucleus="13C", b0=600.0, hi=200.0, lo=0.0, echo=0.5)
    PRESETS = {
        "C′ Q3 180°, carrier on Cα": dict(
            C13, carrier=56.0, band_hi=70.0, band_lo=40.0, rows=[q3(176.0)],
            note="A 0.35 ms Q3 180° on C′ (176 ppm), played as a phase ramp from a carrier "
                 "on Cα (56 ppm), as in triple-resonance experiments: Cα is not excited, "
                 "yet it turns by about −97°."),
        "C′ Q3 180°, carrier on C′": dict(
            C13, carrier=176.0, band_hi=70.0, band_lo=40.0, rows=[q3(176.0)],
            note="The same pulse with the carrier on C′: φBS is the same, because it is "
                 "measured against a delay of the same length; the correction in "
                 "Compensation is not."),
        "Cα Q3 180°, seen on C′": dict(
            C13, carrier=56.0, band_hi=180.0, band_lo=170.0, rows=[q3(56.0)],
            note="The mirror case: a Cα-selective 180° turns the C′ magnetization (170–180 "
                 "ppm) by about as much, with the opposite sign."),
        "Build your own": dict(
            C13, carrier=56.0, band_hi=70.0, band_lo=40.0, rows=[q3(176.0)],
            note="Add segments with + Add segment: φBS is the phase of the whole sequence "
                 "against a delay of the same total length."),
    }
    COMMON = dict(figure="Phase vs shift", n=801, sweep="none", values="", figsave="no",
                  data="no")
    S = {"description_width": "initial"}
    def W(px): return widgets.Layout(width=px)
    def L(text, px): return widgets.Label(text, layout=W(px))
    out = widgets.Output()

    w_preset  = widgets.Dropdown(options=list(PRESETS), description="preset:", style=S,
                                 layout=W("260px"))
    preset_note = widgets.HTML()
    w_figure  = widgets.Dropdown(options=FIGURES, description="figure:", style=S, layout=W("230px"))
    w_nucleus = widgets.Dropdown(options=NUCLEI, description="nucleus:", style=S, layout=W("140px"))
    w_b0      = widgets.FloatText(description="B0 (¹H MHz):", style=S, layout=W("160px"))
    w_carrier = widgets.FloatText(description="carrier (ppm):", style=S, layout=W("160px"),
                                  tooltip="receiver frequency; the rows' 'at' is each pulse's own")
    w_hi      = widgets.FloatText(description="from (ppm):", style=S, layout=W("140px"))
    w_lo      = widgets.FloatText(description="to (ppm):", style=S, layout=W("125px"))
    w_n       = widgets.IntText(description="points:", style=S, layout=W("120px"))
    w_bhi     = widgets.FloatText(description="band from (ppm):", style=S, layout=W("170px"))
    w_blo     = widgets.FloatText(description="to (ppm):", style=S, layout=W("125px"))
    w_echo    = widgets.FloatText(description="echo half (ms):", style=S, layout=W("160px"),
                                  tooltip="method 1: each half of the echo, pulse - d - 180 - d - pulse")
    w_sweep   = widgets.Dropdown(options=list(SWEEPS), description="compare:", style=S,
                                 layout=W("270px"))
    w_values  = widgets.Text(description="values:", style=S, layout=W("220px"),
                             placeholder="e.g. 0.35, 0.7, 1.4")
    w_figsave = widgets.Dropdown(options=FIG_FORMATS, description="figures:", style=S,
                                 layout=W("190px"))
    w_data    = widgets.Dropdown(options=DATA_FORMATS, description="data:", style=S,
                                 layout=W("140px"))
    compare_box, echo_box = widgets.HBox([w_sweep, w_values]), widgets.HBox([w_echo])

    def _sync(change=None):
        phase_fig = w_figure.value == "Phase vs shift"
        compare_box.layout.display = "" if phase_fig else "none"
        echo_box.layout.display = "none" if phase_fig else ""

    # ---- one row per segment (module 1's rows, plus each pulse's own frequency) ---------
    _rows = []
    seg_box = widgets.VBox([])

    def _refresh():
        for n, r in enumerate(_rows, start=1):
            r["idx"].value = str(n)
            r["up"].disabled = (n == 1)
            r["down"].disabled = (n == len(_rows))
        seg_box.children = tuple(r["box"] for r in _rows)

    def _move(row, step):
        i = _rows.index(row)
        j = i + step
        if 0 <= j < len(_rows):
            _rows[i], _rows[j] = _rows[j], _rows[i]
            _refresh()

    def _make_row(kind="shaped pulse", flip=180.0, phase="x", dur=0.35, shape="gausscasq3",
                  pts=500, tr=False, at=176.0, align=0.5):
        idx  = L("", "26px")
        w_kd = widgets.Dropdown(options=SEG_KINDS, value=kind, layout=W("120px"))
        w_fl = widgets.FloatText(value=flip, layout=W("75px"))
        w_ph = widgets.Dropdown(options=list(PHASE_AXIS), value=phase, layout=W("60px"))
        w_du = widgets.FloatText(value=dur, layout=W("80px"))
        w_sh = widgets.Dropdown(options=SHAPES, value=shape, layout=W("130px"))
        w_pt = widgets.IntText(value=pts, layout=W("75px"))
        w_tr = widgets.Checkbox(value=tr, indent=False, layout=W("40px"),
                                tooltip="time-reversed: play this shape backwards")
        w_at = widgets.FloatText(value=at, layout=W("90px"),
                                 tooltip="the frequency of this pulse: its own offset")
        w_al = widgets.Dropdown(options=ALIGNS, value=align, layout=W("90px"),
                                tooltip="where the phase ramp of a shifted pulse is zero "
                                        "(Bruker SPOAL 0 / 0.5 / 1)")
        w_up = widgets.Button(description="↑", tooltip="move this row up", layout=W("32px"))
        w_dn = widgets.Button(description="↓", tooltip="move this row down", layout=W("32px"))
        w_rm = widgets.Button(description="✕", tooltip="remove this row", layout=W("36px"))

        def _sync_row(change=None):
            k = w_kd.value
            w_fl.disabled = w_ph.disabled = w_pt.disabled = w_at.disabled = w_al.disabled = (k == "delay")
            w_sh.disabled = w_tr.disabled = (k != "shaped pulse")
        w_kd.observe(_sync_row, names="value")
        _sync_row()

        def _remove(_b):
            if len(_rows) == 1:
                with out:
                    clear_output(wait=True)
                    print("A sequence needs at least one row.")
                return
            _rows.remove(row)
            _refresh()
        w_rm.on_click(_remove)
        w_up.on_click(lambda _b: _move(row, -1))
        w_dn.on_click(lambda _b: _move(row, +1))
        row = {"idx": idx, "kind": w_kd, "flip": w_fl, "phase": w_ph, "dur": w_du, "shape": w_sh,
               "pts": w_pt, "tr": w_tr, "at": w_at, "align": w_al, "up": w_up, "down": w_dn,
               "box": widgets.HBox([idx, w_kd, w_fl, w_ph, w_du, w_sh, w_pt, w_tr, w_at, w_al,
                                    w_up, w_dn, w_rm])}
        return row

    def _add(_b=None):
        if _rows:                                   # a new row copies the last one
            p = _rows[-1]
            _rows.append(_make_row(*(p[k].value for k in ("kind", "flip", "phase", "dur", "shape",
                                                         "pts", "tr", "at", "align"))))
        else:
            _rows.append(_make_row())
        _refresh()

    def _load_preset(change=None):
        d = {**COMMON, **PRESETS[w_preset.value]}
        _rows.clear()
        for a in d["rows"]:
            _rows.append(_make_row(*a))
        _refresh()
        for w, k in ((w_nucleus, "nucleus"), (w_b0, "b0"), (w_carrier, "carrier"), (w_hi, "hi"),
                     (w_lo, "lo"), (w_n, "n"), (w_bhi, "band_hi"), (w_blo, "band_lo"),
                     (w_echo, "echo"), (w_sweep, "sweep"), (w_values, "values"),
                     (w_figsave, "figsave"), (w_data, "data")):
            w.value = d[k]
        if change is None:                  # first call: open on the default figure
            w_figure.value = d["figure"]
        preset_note.value = f"<i>{d['note']}</i>"
        _sync()
        if change is not None:
            with out:
                clear_output(wait=True)
                print(f"Loaded preset: {w_preset.value}")

    w_figure.observe(_sync, names="value")
    w_preset.observe(_load_preset, names="value")

    # ---- building blocks -------------------------------------------------------
    def _params():
        rows = [dict(kind=r["kind"].value, flip=r["flip"].value, phase=r["phase"].value,
                     dur=r["dur"].value, shape=r["shape"].value, pts=int(r["pts"].value),
                     tr=r["tr"].value, at=r["at"].value, align=r["align"].value) for r in _rows]
        if not any(r["kind"] != "delay" for r in rows):
            raise ValueError("the sequence needs at least one pulse")
        return dict(nucleus=w_nucleus.value, b0=w_b0.value, carrier=w_carrier.value,
                    echo=w_echo.value, rows=rows)

    def _first(p):
        """Index of the first pulse row: the one compare and Compensation act on."""
        return next(i for i, r in enumerate(p["rows"]) if r["kind"] != "delay")

    def _with_first(p, key, v):
        rows = [dict(r) for r in p["rows"]]
        rows[_first(p)][key] = v
        return {**p, "rows": rows}

    def _mhz(p):
        return nucleus_mhz(p["b0"], p["nucleus"])

    def _element(r, p, ramp=True):
        if r["kind"] == "delay":
            return Delay(r["dur"])
        if r["kind"] == "hard pulse":
            shape = RFShape.create("hard", duration=r["dur"], points=max(2, r["pts"]))
        else:
            shape = RFShape.create(r["shape"], duration=r["dur"], points=r["pts"],
                                   time_reversed=bool(r["tr"]))
        flip = None if shape.calibration_mode == "adiabatic" else np.deg2rad(r["flip"])
        off = ppm_to_khz(r["at"], p["carrier"], _mhz(p)) if ramp else 0.0
        return OffsetPulse(shape, flip, axis=PHASE_AXIS[r["phase"]],
                           backend=NumpyBackend(Gamma=gyro_ratio(p["nucleus"])),
                           offset_khz=off, align=r["align"])

    def _elements(p, ramp=True):
        return [_element(r, p, ramp) for r in p["rows"]]

    def _length(els):
        return sum(e.duration if isinstance(e, Delay) else e.shape.duration for e in els)

    def _ramp_note(p):
        offs = [ppm_to_khz(r["at"], p["carrier"], _mhz(p)) for r in p["rows"] if r["kind"] != "delay"]
        if any(abs(o) > 1e-9 for o in offs):
            return "  (pulses drawn without their phase ramps: " + ", ".join(f"{o:+.2f}" for o in offs) + " kHz)"
        return ""

    def _hard(flip, phase, p):
        return hard_pulse(flip, phase, nu1_khz=flip / 360.0 / 0.001, nucleus=p["nucleus"])  # 1 us

    def _run(elements, df, start="x"):
        return PulseSequence(list(elements)).run(magnetization(len(df), start), df)

    def _rel(A, B):
        """Transverse phase of A relative to B, deg, in (-180, 180]."""
        return np.degrees(np.angle((A[0] + 1j * A[1]) / (B[0] + 1j * B[1])))

    def _unwrap_out(phi, D):
        """Unwrap each side of the pulse frequency starting far out (small phase)."""
        order = np.argsort(D)
        ph, Ds, out_ = np.asarray(phi, float)[order], D[order], np.empty(len(phi))
        neg, pos = Ds < 0, Ds >= 0
        res = ph.copy()
        res[neg] = np.degrees(np.unwrap(np.radians(ph[neg])))
        res[pos] = np.degrees(np.unwrap(np.radians(ph[pos][::-1])))[::-1]
        out_[order] = res
        return out_

    def _phase_vs_shift(p, ppm):
        df = ppm_to_khz(ppm, p["carrier"], _mhz(p))          # kHz from the carrier
        els = _elements(p)
        P0 = els[_first(p)]
        D = df - P0.offset_khz                                # kHz from the first pulse
        phi = _unwrap_out(_rel(_run(els, df), _run([Delay(_length(els))], df)), D)
        excited = _run(els, df, "z")[2] < 0.99
        # the formulas, summed over the pulses (valid where none of them excites)
        g = gyro_ratio(p["nucleus"])
        integral, approx = np.zeros(len(df)), np.zeros(len(df))
        for P in els:
            if isinstance(P, Delay):
                continue
            nu1 = abs(g) * np.abs(P.calibrated_rf())          # kHz
            Dp = df - P.offset_khz
            sgn = np.sign(Dp) * np.sign(g)
            Da = np.abs(Dp)[:, None]
            integral += sgn * 360 * np.sum((np.sqrt(Da ** 2 + nu1 ** 2) - Da) * P.shape.dt, axis=1)
            with np.errstate(divide="ignore", invalid="ignore"):
                approx += sgn * 360 * np.sum(nu1 ** 2 * P.shape.dt) / (2 * np.abs(Dp))
        return dict(phi=np.where(excited, np.nan, phi), excited=excited,
                    integral=np.where(excited, np.nan, integral),
                    approx=np.where(excited, np.nan, approx), pulse=P0, D=D, elements=els)

    def _compensation(p, ppm):
        df = ppm_to_khz(ppm, p["carrier"], _mhz(p))
        P = _element(p["rows"][_first(p)], p)
        tau, d = P.shape.duration, p["echo"]
        h180 = _hard(180.0, "x", p)
        # method 1: the identical pulse at the mirror position of an echo
        REF = [Delay(tau), Delay(d), h180, Delay(d), Delay(tau)]
        ONE = [P, Delay(d), h180, Delay(d), Delay(tau)]
        TWO = [P, Delay(d), h180, Delay(d), _element(p["rows"][_first(p)], p)]
        m1_one, m1_two = (_rel(_run(s, df), _run(REF, df)) for s in (ONE, TWO))
        # methods 2 + 3: phi_BS over the band ~ a constant + a slope in offset;
        # the slope is what a delay does, the constant is a phase
        phi = np.degrees(np.unwrap(np.radians(_rel(_run([P], df), _run([Delay(tau)], df))[
            np.argsort(df)])))
        dfs = np.sort(df)
        dc = 0.5 * (dfs[0] + dfs[-1])
        slope = np.polyfit(dfs - dc, phi, 1)[0]                                   # deg / kHz
        free = _run([Delay(1.0)], dfs)
        per_ms = np.polyfit(dfs, np.degrees(np.unwrap(np.angle(free[0] + 1j * free[1]))), 1)[0]
        t_eq = slope / per_ms                                 # ms of delay the pulse acts like
        T = max(0.12, 2 * abs(t_eq) + 0.05)                   # a free delay after the pulse
        ref, unc, dly = (_run(s, df) for s in ([Delay(tau), Delay(T)], [P, Delay(T)],
                                               [P, Delay(T - t_eq)]))
        e_unc, e_dly = _rel(unc, ref), _rel(dly, ref)
        const = float(e_dly[np.argmin(np.abs(df - dc))])
        e_fix = (e_dly - const + 180.0) % 360.0 - 180.0
        # executed: a 90 deg readout (phase y; y + const when corrected) stores the phase on z
        y = np.pi / 2
        h90 = lambda ph: Pulse(RFShape.create("hard", duration=0.001, points=10), np.pi / 2,
                               axis=ph, backend=NumpyBackend(Gamma=gyro_ratio(p["nucleus"])))
        mz_ref = _run([Delay(tau), Delay(T), h90(y)], df)[2]
        mz_fix = _run([P, Delay(T - t_eq), h90(y + np.radians(const))], df)[2]
        mz_unc = _run([P, Delay(T), h90(y)], df)[2]
        flat = _element(p["rows"][_first(p)], p, ramp=False)          # for the diagrams
        return dict(m1_one=m1_one, m1_two=m1_two, e_unc=e_unc, e_dly=e_dly, e_fix=e_fix,
                    t_eq=t_eq, const=const, T=T,
                    seq1=[flat, Delay(d), h180, Delay(d), flat],
                    seq23=[flat, Delay(T - t_eq), h90(y + np.radians(const))],
                    mz_err_fix=np.abs(mz_fix - mz_ref).max(),
                    mz_err_unc=np.abs(mz_unc - mz_ref).max(),
                    excited=(_run([P], df, "z")[2] < 0.99).any())

    # ---- figures --------------------------------------------------------------
    def _marks(ax, p, band=True, label=True):
        ax.axhline(0, color="0.8", lw=0.5)
        ax.axvline(p["carrier"], color="0.4", ls=":", lw=0.9)
        ats = sorted({r["at"] for r in p["rows"] if r["kind"] != "delay"})
        for a in ats:
            ax.axvline(a, color="0.4", ls="--", lw=0.9)
        if band:
            ax.axvspan(w_blo.value, w_bhi.value, color="#1f77b4", alpha=0.07, lw=0)
        if label:
            tr = ax.get_xaxis_transform()
            ax.text(p["carrier"], 0.97, " carrier", transform=tr, ha="left", va="top", color="0.3")
            for a in ats:
                ax.text(a, 0.88, " pulse", transform=tr, ha="left", va="top", color="0.3")
            if band:
                ax.text(0.5 * (w_blo.value + w_bhi.value), 0.03, "band", transform=tr,
                        ha="center", va="bottom", color="#1f77b4")

    def _deg_axis(ax):
        ax.set(ylim=(-180, 180), yticks=np.arange(-180, 181, 60))

    def _nuc_axis_label(p):
        return f"{nuc_label(p['nucleus'])} chemical shift (ppm)"

    def _run_button(_b):
        with out:
            clear_output(wait=True)
            try:
                print("Run button has been clicked. Now running... Please wait...")
                base = _params()
                print(f"  {nuc_label(base['nucleus'])} at {_mhz(base):.3f} MHz, carrier "
                      f"{base['carrier']:g} ppm")
                for i, (r, e) in enumerate(zip(base["rows"], _elements(base)), start=1):
                    if isinstance(e, Delay):
                        print(f"  {i}. delay {r['dur']:g} ms")
                    else:
                        print(f"  {i}. {r['shape'] if r['kind'] == 'shaped pulse' else 'hard'}"
                              f"{' (tr)' if r['tr'] and r['kind'] == 'shaped pulse' else ''} "
                              f"{r['flip']:g}°{r['phase']}, {r['dur']:g} ms, nu1_max "
                              f"{e.nu1_max:.3f} kHz, at {r['at']:g} ppm "
                              f"({e.offset_khz:+.3f} kHz from the carrier)")
                stem = "pulsim_bloch_siegert_" + w_figure.value.split()[0].lower()
                with save_figures(w_figsave.value, stem):
                    if w_figure.value == "Phase vs shift":
                        _figure_phase(base)
                    else:
                        _figure_compensation(base)
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")
            finally:
                flush_downloads()      # last: nothing may print after a Colab download

    def _figure_phase(base):
        key = SWEEPS[w_sweep.value]
        values = parse_values(w_values.value) if key else []
        if key and not values:
            raise ValueError("type the values to compare, e.g. 0.35, 0.7, 1.4")
        name, unit = (w_sweep.value[:-1].split(" (") if key else ("", ""))
        name = name.replace("first pulse: ", "")
        if not key:
            runs = [("", base)]
        elif key == "b0":
            runs = [(f"B0 = {v:g} MHz", {**base, "b0": v}) for v in values]
        else:
            runs = [(f"{name} = {v:g} {unit}", _with_first(base, key, v)) for v in values]
        ppm = np.linspace(w_hi.value, w_lo.value, int(w_n.value))
        t0 = time.time()
        res = [(lab, p, _phase_vs_shift(p, ppm)) for lab, p in runs]
        print(f"  {len(res)} run(s) x {len(ppm)} shifts in {time.time() - t0:.2f} s")
        bc = 0.5 * (w_bhi.value + w_blo.value)
        ib, iedge = int(np.argmin(np.abs(ppm - bc))), [int(np.argmin(np.abs(ppm - v)))
                                                     for v in (w_bhi.value, w_blo.value)]
        def deg(v):
            return "excited (not BS)" if np.isnan(v) else f"{v:+.1f}°"
        for lab, p, r in res:
            exc = ppm[r["excited"]]
            ex = f"excites {exc.max():.1f}..{exc.min():.1f} ppm" if exc.size else "excites nothing here"
            print(f"  {lab or 'result'}: {ex}; phi_BS at the band center ({bc:g} ppm) "
                  f"{deg(r['phi'][ib])}, at its edges {deg(r['phi'][iedge[0]])} / "
                  f"{deg(r['phi'][iedge[1]])}  (w1^2/(2 Delta): {deg(r['approx'][ib])})")
        show_sequence(_elements(base, ramp=False), nucleus=base["nucleus"],
                      title="run A: the sequence;  run B: a delay of the same length;  "
                            "φ$_{BS}$ = phase(A) − phase(B)" + _ramp_note(base))
        fig, ax = plt.subplots(figsize=(FIG_W, 3.4))
        for k, (lab, p, r) in enumerate(res):
            ax.fill_between(ppm, -180, 180, where=r["excited"], color="0.88", lw=0,
                            label="excited by the pulses\n(not Bloch–Siegert)" if k == 0 else None)
        if not key:
            r = res[0][2]
            ax.plot(ppm, r["phi"], color="k", lw=1.6, label="PULSIM: sequence − delay")
            ax.plot(ppm, r["integral"], color="C0", lw=0.9,
                    label=r"$\int(\sqrt{\Delta^2+\omega_1^2}-|\Delta|)\,dt$")
            ax.plot(ppm, r["approx"], color="C1", lw=0.9, ls="--",
                    label=r"$\frac{1}{2\Delta}\int\omega_1^2\,dt$")
        else:
            for k, (lab, p, r) in enumerate(res):
                ax.plot(ppm, r["phi"], color=PALETTE[k % len(PALETTE)], lw=1.3, label=lab)
        _marks(ax, base)
        ax.set_xlim(w_hi.value, w_lo.value)
        _deg_axis(ax)
        ax.set(xlabel=_nuc_axis_label(base), ylabel=r"$\phi_{BS}$ (deg)")
        if key not in ("at", "b0"):               # one pulse frequency for all runs
            a0 = base["rows"][_first(base)]["at"]
            f = lambda x: (x - a0) * _mhz(base) / 1000.0
            g = lambda kk: kk * 1000.0 / _mhz(base) + a0
            ax.secondary_xaxis("top", functions=(f, g)).set_xlabel("offset from the first pulse (kHz)")
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
        fig.suptitle("Bloch–Siegert phase of the sequence")
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            cols = {"ppm": ppm}
            if key not in ("at", "b0"):
                cols["offset from the first pulse (kHz)"] = res[0][2]["D"]
            for lab, p, r in res:
                cols[f"phi_BS deg [{lab}]" if lab else "phi_BS deg"] = r["phi"]
            if not key:
                cols["effective-field integral deg"] = res[0][2]["integral"]
                cols["w1^2/(2 Delta) deg"] = res[0][2]["approx"]
            save_csv("pulsim_bloch_siegert_phase", cols)

    def _figure_compensation(base):
        if w_bhi.value == w_blo.value:
            raise ValueError("the band needs two different ends")
        ppm = np.linspace(max(w_bhi.value, w_blo.value), min(w_bhi.value, w_blo.value),
                          int(w_n.value))
        print(f"  correcting the pulse of row {_first(base) + 1}")
        c = _compensation(base, ppm)
        if c["excited"]:
            print("  WARNING: the pulse excites part of the band -- that is not a "
                  "Bloch-Siegert phase, and no correction below applies to it")
        print(f"  method 1 (identical pulse mirrored in a {base['echo']:g} ms + "
              f"{base['echo']:g} ms echo): one pulse {np.nanmin(c['m1_one']):+.1f}..."
              f"{np.nanmax(c['m1_one']):+.1f}°, with the mirror pulse max |error| "
              f"{np.abs(c['m1_two']).max():.2f}°")
        print(f"  method 2: the pulse acts like {c['t_eq'] * 1000:+.1f} µs of delay -> "
              f"LENGTHEN the next delay by {-c['t_eq'] * 1000:+.1f} µs")
        print(f"  method 3: then shift the phase of the next pulse (or the receiver) by "
              f"{c['const']:+.1f}°; residual max |error| {np.abs(c['e_fix']).max():.2f}°")
        print(f"  executed with a 90° readout: max |Mz(corrected) - Mz(no pulse)| = "
              f"{c['mz_err_fix']:.3f} (uncorrected {c['mz_err_unc']:.2f})")
        note = _ramp_note({**base, "rows": [base["rows"][_first(base)]]})
        show_sequence(c["seq1"], nucleus=base["nucleus"],
                      title="method 1: pulse – d – 180° – d – identical pulse" + note)
        show_sequence(c["seq23"], nucleus=base["nucleus"],
                      title=f"methods 2 + 3: pulse – delay {-c['t_eq'] * 1000:+.1f} µs longer – "
                            f"next pulse {c['const']:+.0f}°" + note)
        fig, axs = plt.subplots(1, 2, figsize=(FIG_W, 3.0), sharey=True)
        axs[0].plot(ppm, c["m1_one"], color="k", lw=1.3, label="one pulse")
        axs[0].plot(ppm, c["m1_two"], color="C0", lw=1.3, label="+ identical pulse, mirrored")
        axs[0].set_title("method 1")
        axs[1].plot(ppm, c["e_unc"], color="k", lw=1.3, label="no correction")
        axs[1].plot(ppm, c["e_dly"], color="C0", lw=1.3,
                    label=f"delay {-c['t_eq'] * 1000:+.1f} µs")
        axs[1].plot(ppm, c["e_fix"], color="C1", lw=1.3, ls="--",
                    label=f"+ next pulse {c['const']:+.0f}°")
        axs[1].set_title("methods 2 + 3")
        for a in axs:
            a.axhline(0, color="0.8", lw=0.5)
            a.set_xlim(ppm[0], ppm[-1])
            _deg_axis(a)
            a.set_xlabel(_nuc_axis_label(base))
            a.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=False)
        axs[0].set_ylabel("phase error (deg)")
        fig.suptitle(f"Removing the Bloch–Siegert phase over {ppm[0]:g}–{ppm[-1]:g} ppm")
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            save_csv("pulsim_bloch_siegert_compensation",
                     {"ppm": ppm, "method 1, one pulse (deg)": c["m1_one"],
                      "method 1, mirrored (deg)": c["m1_two"],
                      "no correction (deg)": c["e_unc"],
                      f"delay {-c['t_eq'] * 1000:+.1f} us (deg)": c["e_dly"],
                      f"+ phase {c['const']:+.1f} deg (deg)": c["e_fix"]})

    b_add   = widgets.Button(description="+ Add segment", layout=W("130px"))
    b_run   = widgets.Button(description="Run", button_style="primary", layout=W("110px"))
    b_reset = widgets.Button(description="Reload preset", layout=W("120px"),
                             tooltip="restore the selected preset (undo your edits)")
    b_add.on_click(_add)
    b_run.on_click(_run_button)
    b_reset.on_click(lambda _b: _load_preset(_b))
    _load_preset()

    header = widgets.HBox([L("#", "26px"), L("segment", "120px"), L("flip °", "75px"),
                           L("phase", "60px"), L("dur ms", "80px"), L("shape", "130px"),
                           L("points", "75px"), L("TR", "40px"), L("at (ppm)", "90px"),
                           L("ramp zero", "90px")])
    display(widgets.VBox([
        widgets.HBox([w_preset, preset_note]),
        separator(),
        widgets.HTML("<b>Segments</b> — applied in order; <i>at</i> is each pulse's own "
                     "frequency, applied as a phase ramp; TR = time-reversed shaped pulse"),
        header, seg_box,
        separator(),
        widgets.HTML("<b>Spectrometer</b> — shifts in ppm"),
        widgets.HBox([w_nucleus, w_b0, w_carrier]),
        widgets.HBox([w_hi, w_lo, w_n]),
        separator(),
        widgets.HTML("<b>Band to correct</b>"),
        widgets.HBox([w_bhi, w_blo]),
        separator(),
        widgets.HTML("<b>Figure</b>"),
        widgets.HBox([w_figure, compare_box, echo_box]),
        separator(),
        widgets.HBox([b_add, b_run, b_reset, w_figsave, w_data]),
        out,
    ]))


bloch_siegert_widget()

In [ ]:
#@title 3. Relaxation-aware delays { display-mode: "form" }
from PULSIM.spin_operators import SpinOperators
from PULSIM.liouville import IdealPulse, SimultaneousPulses
from PULSIM.relaxation import Relaxation
from PULSIM.pulse_diagnostics import optimize_delay


def relaxation_widget():
    import re
    PI = np.pi
    NUCLEI = ["H", "13C", "15N", "19F", "31P"]
    NUC_OPTS = [(nuc_label(n), n) for n in NUCLEI]
    SEG_KINDS = ["ideal pulse", "delay", "shaped pulse"]   # as in the main notebook
    PHASES = ["x", "y", "-x", "-y"]
    ALIGNS = ["left", "center", "right"]
    SHAPES = sorted(s for s in RFShape.available() if s not in ("file", "composite", "varian"))
    FIGURES = ["Delay optimization", "Optimum vs T₂", "Product operators"]

    # ---- presets (rows as in the main notebook: kind, flip, phase, channel, ms[, options])
    def _kHz(J, pair=(0, 1)):
        return J.get(pair, 0.0) / 1000.0

    def inept_rows(S, d):
        return [("ideal pulse", 90.0, "x", "H", 0.0), ("delay", 0.0, "x", "H", d),
                ("ideal pulse", 180.0, "x", "H", 0.0), ("ideal pulse", 180.0, "x", S, 0.0),
                ("delay", 0.0, "x", "H", d),
                ("ideal pulse", 90.0, "y", "H", 0.0, dict(align="right")),
                ("ideal pulse", 90.0, "x", S, 0.0, dict(align="right"))]

    def f_inept(J, T1, T2, D):
        """INEPT, transfer to 2IzSy: exact when S has no T1 (Sz during the delays)."""
        if T1[1] is not None:
            return None
        Jk = _kHz(J)
        dec = (lambda d: 1.0) if T2[0] is None else (lambda d: np.exp(-2 * d / T2[0]))
        opt = 1 / (4 * Jk) if T2[0] is None else np.arctan(PI * Jk * T2[0]) / (2 * PI * Jk)
        return dict(op="2IzSy", curve=lambda d: -np.sin(2 * PI * Jk * d) * dec(d), opt=opt,
                    label=r"$-\sin(2\pi J\Delta)\,e^{-2\Delta/T_2(I)}$",
                    opt_label=r"$\arctan(\pi J T_2(I))/(2\pi J)$")

    def f_refocused(J, T1, T2, D):
        """Refocused INEPT, Delta2 swept: exact when neither spin has T1."""
        if T1[0] is not None or T1[1] is not None:
            return None
        Jk = _kHz(J)
        a = np.sin(2 * PI * Jk * D[1]) * (1.0 if T2[0] is None else np.exp(-2 * D[1] / T2[0]))
        dec = (lambda d: 1.0) if T2[1] is None else (lambda d: np.exp(-2 * d / T2[1]))
        opt = 1 / (4 * Jk) if T2[1] is None else np.arctan(PI * Jk * T2[1]) / (2 * PI * Jk)
        return dict(op="Sx", curve=lambda d: a * np.sin(2 * PI * Jk * d) * dec(d), opt=opt,
                    label=r"$\sin(2\pi J\Delta_1)e^{-2\Delta_1/T_2(I)}\,\sin(2\pi J\Delta_2)e^{-2\Delta_2/T_2(S)}$",
                    opt_label=r"$\arctan(\pi J T_2(S))/(2\pi J)$")

    QUARTER = (lambda J, D: 1 / (4 * _kHz(J)), "1/(4J)")
    J_NH, J_CH = 92.0, 140.0
    D_NH, D_CH = 1 / (4 * J_NH / 1000), 1 / (4 * J_CH / 1000)
    PRESETS = {
        "INEPT (1H-15N)": dict(
            spins=[("H", 0.0, "", "10"), ("15N", 0.0, "", "")], J={(0, 1): J_NH},
            rows=inept_rows("15N", D_NH), start="Iz", read="2IzSy", dmax=2 * D_NH,
            sweep={"Δ (rows 2 + 5)": dict(rows=[1, 4], ref=QUARTER, formula=f_inept)},
            note="1H T2 = 10 ms: the best delay is shorter than 1/(4J). Set T1(15N) too and "
                 "the optimum moves again; T1(1H) and T2(15N) do not move it."),
        "Shaped INEPT (1H-15N)": dict(
            spins=[("H", 0.0, "", "10"), ("15N", 0.0, "", "")], J={(0, 1): J_NH},
            rows=[("shaped pulse", 90.0, "x", "H", 1.5, dict(shape="eburp2", points=500)),
                  ("delay", 0.0, "x", "H", 2.70),
                  ("shaped pulse", 180.0, "x", "H", 0.02, dict(shape="hard", points=20, align="center")),
                  ("shaped pulse", 180.0, "x", "15N", 0.08, dict(shape="hard", points=40, align="center")),
                  ("delay", 0.0, "x", "H", 2.70),
                  ("shaped pulse", 90.0, "y", "H", 1.5, dict(shape="eburp2", points=500, tr=True, align="right")),
                  ("shaped pulse", 90.0, "x", "15N", 0.04, dict(shape="hard", points=20, align="right"))],
            start="Iz", read="2IzSy", dmax=2 * D_NH,
            sweep={"Δ (rows 2 + 5)": dict(rows=[1, 4], ref=QUARTER, formula=None)},
            note="Example 4's shaped INEPT (E-BURP-2 forward in, time-reversed out) with 1H "
                 "T2 = 10 ms: the hidden delays of the shaped pulses and relaxation set the "
                 "best delay together."),
        "Refocused INEPT (1H-15N)": dict(
            spins=[("H", 0.0, "", "10"), ("15N", 0.0, "", "30")], J={(0, 1): J_NH},
            rows=inept_rows("15N", D_NH) + [("delay", 0.0, "x", "H", D_NH),
                                            ("ideal pulse", 180.0, "x", "H", 0.0),
                                            ("ideal pulse", 180.0, "x", "15N", 0.0),
                                            ("delay", 0.0, "x", "H", D_NH)],
            start="Iz", read="Sx", dmax=2 * D_NH,
            sweep={"Δ2 refocusing (rows 8 + 11)": dict(rows=[7, 10], ref=QUARTER, formula=f_refocused),
                   "Δ1 INEPT (rows 2 + 5)": dict(rows=[1, 4], ref=QUARTER, formula=None)},
            note="During Δ2 the 15N coherence relaxes (T2 of 15N): the best refocusing delay "
                 "is set by T2(15N), not by T2(1H)."),
        "INEPT (1H-13C)": dict(
            spins=[("H", 0.0, "", "20"), ("13C", 0.0, "", "")], J={(0, 1): J_CH},
            rows=inept_rows("13C", D_CH), start="Iz", read="2IzSy", dmax=2 * D_CH,
            sweep={"Δ (rows 2 + 5)": dict(rows=[1, 4], ref=QUARTER, formula=f_inept)},
            note="1J(CH) = 140 Hz is large, so the delays are short and 1H T2 = 20 ms costs "
                 "little: the optimum stays close to 1/(4J)."),
        "Build your own": dict(
            spins=[("H", 0.0, "", ""), ("13C", 0.0, "", "")], J={(0, 1): J_CH},
            rows=[("ideal pulse", 90.0, "x", "H", 0.0), ("delay", 0.0, "x", "H", 1.0)],
            start="Iz", read="Ix, Iy, Iz", dmax=2.0, sweep={},
            note="Start from a 90° and a delay: set the spins, relaxation times and couplings, "
                 "then add segments with + Add segment."),
    }
    DEFAULTS = dict(dmin=0.0, dn=121, t2_from=2.0, t2_to=200.0, t2_n=15,
                    sweep="none", values="", figsave="no", data="no")

    S_ = {"description_width": "initial"}
    def W(px): return widgets.Layout(width=px)
    def L(text, px): return widgets.Label(text, layout=W(px))
    out = widgets.Output()

    # ---- spin system: nucleus, offset, T1, T2 per spin; J per pair -------------------
    w_nspins = widgets.IntSlider(value=2, min=1, max=3, description="spins:", style=S_,
                                 layout=W("200px"))
    spin_box, coupling_box = widgets.VBox([]), widgets.VBox([])
    _spins, _couplings = [], []

    def _rebuild_system(change=None):
        n = w_nspins.value
        while len(_spins) < n:
            i = len(_spins)
            _spins.append({
                "nucleus": widgets.Dropdown(options=NUC_OPTS, value="H" if i == 0 else "13C",
                                            layout=W("90px")),
                "offset": widgets.FloatText(value=0.0, layout=W("90px")),
                "T1": widgets.Text(placeholder="none", layout=W("90px")),
                "T2": widgets.Text(placeholder="none", layout=W("90px"))})
        del _spins[n:]
        spin_box.children = tuple(
            widgets.HBox([L(f"spin {i} ({'ISK'[i]})", "90px"), s["nucleus"], s["offset"],
                          s["T1"], s["T2"]]) for i, s in enumerate(_spins))
        pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
        old = {p["pair"]: p["J"].value for p in _couplings}
        _couplings.clear()
        for (i, j) in pairs:
            _couplings.append({"pair": (i, j),
                               "J": widgets.FloatText(value=old.get((i, j), 0.0), layout=W("90px"))})
        coupling_box.children = tuple(widgets.HBox([L(f"J({i},{j}) Hz", "90px"), c["J"]])
                                      for c, (i, j) in zip(_couplings, pairs))
        _sync_compare()
        _sync_t2_spin()

    w_nspins.observe(_rebuild_system, names="value")

    # ---- figure controls --------------------------------------------------------------
    w_figure = widgets.Dropdown(options=FIGURES, description="figure:", style=S_, layout=W("230px"))
    w_start  = widgets.Text(description="start:", style=S_, layout=W("160px"))
    w_read   = widgets.Text(description="read out:", style=S_, layout=W("260px"))
    w_dsel   = widgets.Dropdown(description="sweep:", style=S_, layout=W("250px"))
    w_dmin   = widgets.FloatText(description="from (ms):", style=S_, layout=W("150px"))
    w_dmax   = widgets.FloatText(description="to (ms):", style=S_, layout=W("140px"))
    w_dn     = widgets.IntText(description="steps:", style=S_, layout=W("120px"))
    w_sweep  = widgets.Dropdown(options=["none"], description="compare:", style=S_,
                                layout=W("220px"))
    w_values = widgets.Text(description="values:", style=S_, layout=W("220px"),
                            placeholder="e.g. 5, 10, 20")
    w_t2spin = widgets.Dropdown(options=[("spin 0 (I)", 0)], description="T₂ of:", style=S_,
                                layout=W("160px"))
    w_t2from = widgets.FloatText(description="from (ms):", style=S_, layout=W("150px"))
    w_t2to   = widgets.FloatText(description="to (ms):", style=S_, layout=W("140px"))
    w_t2n    = widgets.IntText(description="points:", style=S_, layout=W("120px"))
    w_figsave = widgets.Dropdown(options=FIG_FORMATS, description="figures:", style=S_,
                                 layout=W("200px"))
    w_data   = widgets.Dropdown(options=DATA_FORMATS, description="data:", style=S_,
                                layout=W("140px"))
    delay_box = widgets.HBox([w_dsel, w_dmin, w_dmax, w_dn])
    compare_box = widgets.HBox([w_sweep, w_values])
    t2_box = widgets.HBox([w_t2spin, w_t2from, w_t2to, w_t2n])
    _SHOWN = {"Delay optimization": [delay_box, compare_box],
              "Optimum vs T₂": [delay_box, t2_box],
              "Product operators": []}

    def _show_figure_controls(change=None):
        for b in (delay_box, compare_box, t2_box):
            b.layout.display = "" if any(b is s for s in _SHOWN[w_figure.value]) else "none"
    w_figure.observe(_show_figure_controls, names="value")

    def _sync_compare():
        opts = ["none"] + [f"{t}({'ISK'[i]}) (ms)" for i in range(len(_spins)) for t in ("T₂", "T₁")]
        opts += [f"J({i},{j}) (Hz)" for (i, j) in (c["pair"] for c in _couplings)]
        old = w_sweep.value
        w_sweep.options = opts
        w_sweep.value = old if old in opts else "none"

    def _sync_t2_spin():
        opts = [(f"spin {i} ({'ISK'[i]})", i) for i in range(len(_spins))]
        old = w_t2spin.value
        w_t2spin.options = opts
        w_t2spin.value = old if old in [v for _, v in opts] else 0

    # ---- one row per segment (the main notebook's density-matrix rows) -----------------
    _rows, _preset_rows, _group_rows = [], [], {}
    seg_box = widgets.VBox([])
    w_preset = widgets.Dropdown(options=list(PRESETS), description="preset:", style=S_,
                                layout=W("240px"))
    preset_note = widgets.HTML()

    def _refresh():
        for n, r in enumerate(_rows, start=1):
            r["idx"].value = str(n)
            r["up"].disabled = (n == 1)
            r["down"].disabled = (n == len(_rows))
        seg_box.children = tuple(r["box"] for r in _rows)
        _sync_delay_choices()

    def _move(row, step):
        i = _rows.index(row)
        j = i + step
        if 0 <= j < len(_rows):
            _rows[i], _rows[j] = _rows[j], _rows[i]
            _refresh()

    def _make_row(kind="ideal pulse", flip=90.0, phase="x", channel="H", duration=1.0,
                  shape="gausscasq5", points=500, tr=False, align="center"):
        idx = L("", "26px")
        w_kind = widgets.Dropdown(options=SEG_KINDS, value=kind, layout=W("120px"))
        w_flip = widgets.FloatText(value=flip, layout=W("75px"))
        w_ph   = widgets.Dropdown(options=PHASES, value=phase, layout=W("60px"))
        w_ch   = widgets.Dropdown(options=NUC_OPTS, value=channel, layout=W("80px"))
        w_dur  = widgets.FloatText(value=round(duration, 6), layout=W("80px"))
        w_shp  = widgets.Dropdown(options=SHAPES, value=shape if shape in SHAPES else SHAPES[0],
                                  layout=W("130px"))
        w_pts  = widgets.IntText(value=points, layout=W("75px"))
        w_tr   = widgets.Checkbox(value=tr, indent=False, layout=W("40px"),
                                  tooltip="time-reversed: play the shape backwards (shaped pulses)")
        w_al   = widgets.Dropdown(options=ALIGNS, value=align, layout=W("80px"),
                                  tooltip="shaped pulses played together: where a shorter one "
                                          "sits inside the longest (the first row of the group decides)")
        w_up   = widgets.Button(description="↑", tooltip="move this segment up", layout=W("32px"))
        w_dn   = widgets.Button(description="↓", tooltip="move this segment down", layout=W("32px"))
        w_rm   = widgets.Button(description="✕", tooltip="remove", layout=W("36px"))

        def _sync(change=None):
            k = w_kind.value
            w_flip.disabled = w_ph.disabled = w_ch.disabled = (k == "delay")
            w_shp.disabled = w_pts.disabled = w_tr.disabled = w_al.disabled = (k != "shaped pulse")
            w_dur.disabled = (k == "ideal pulse")
            if change is not None:
                _sync_delay_choices()
        w_kind.observe(_sync, names="value")
        _sync()

        def _remove(_b):
            if len(_rows) == 1:
                with out:
                    clear_output(wait=True)
                    print("A sequence needs at least one segment.")
                return
            _rows.remove(row)
            _refresh()
        w_rm.on_click(_remove)
        w_up.on_click(lambda _b: _move(row, -1))
        w_dn.on_click(lambda _b: _move(row, +1))
        row = {"idx": idx, "kind": w_kind, "flip": w_flip, "phase": w_ph, "channel": w_ch,
               "duration": w_dur, "shape": w_shp, "points": w_pts, "tr": w_tr, "align": w_al,
               "up": w_up, "down": w_dn,
               "box": widgets.HBox([idx, w_kind, w_flip, w_ph, w_dur, w_shp, w_pts, w_tr,
                                    w_ch, w_al, w_up, w_dn, w_rm])}
        return row

    def _add(_b=None):
        if _rows:
            p = _rows[-1]
            _rows.append(_make_row(p["kind"].value, p["flip"].value, p["phase"].value,
                                   p["channel"].value, p["duration"].value, p["shape"].value,
                                   p["points"].value, p["tr"].value, p["align"].value))
        else:
            _rows.append(_make_row())
        _refresh()

    def _sync_delay_choices():
        """The preset's named sweep groups first, then the generic choices."""
        P = PRESETS[w_preset.value]
        opts = [(label, "g:" + label) for label in P.get("sweep", {})
                if label in _group_rows and all(any(r is x for x in _rows) for r in _group_rows[label])]
        delays = [i for i, r in enumerate(_rows) if r["kind"].value == "delay"]
        if delays:
            opts.append(("all delays together", "all"))
        opts += [(f"row {i + 1} only", i) for i in delays]
        opts = opts or [("(no delay rows)", None)]
        old = w_dsel.value
        w_dsel.options = opts
        values = [v for _, v in opts]
        w_dsel.value = old if old in values else values[0]

    # ---- presets ------------------------------------------------------------------------
    def _load_preset(change=None):
        p, d = PRESETS[w_preset.value], DEFAULTS
        w_nspins.value = len(p["spins"])
        _rebuild_system()
        for s_, (nuc, off, t1, t2) in zip(_spins, p["spins"]):
            s_["nucleus"].value, s_["offset"].value = nuc, off
            s_["T1"].value, s_["T2"].value = t1, t2
        for c in _couplings:
            c["J"].value = p["J"].get(c["pair"], 0.0)
        _rows.clear()
        _group_rows.clear()
        for a in p["rows"]:
            _rows.append(_make_row(*a[:5], **(a[5] if len(a) > 5 else {})))
        _preset_rows[:] = _rows
        for label, gr in p.get("sweep", {}).items():
            _group_rows[label] = [_rows[i] for i in gr["rows"]]
        _refresh()
        w_start.value, w_read.value = p["start"], p["read"]
        first = next(iter(p.get("sweep", {})), None)
        if first:
            w_dsel.value = "g:" + first
        w_dmin.value, w_dmax.value, w_dn.value = d["dmin"], round(p["dmax"], 3), d["dn"]
        w_t2from.value, w_t2to.value, w_t2n.value = d["t2_from"], d["t2_to"], d["t2_n"]
        w_sweep.value, w_values.value = d["sweep"], d["values"]
        w_figsave.value, w_data.value = d["figsave"], d["data"]
        _show_figure_controls()
        preset_note.value = f"<i>{p['note']}</i>"
        if change is not None:
            with out:
                clear_output(wait=True)
                print(f"Loaded preset: {w_preset.value}")

    w_preset.observe(_load_preset, names="value")

    # ---- the sequence from the rows (grouped as in the main notebook) --------------------
    def _events():
        """Consecutive pulses of the same kind on different channels play together."""
        groups, i = [], 0
        while i < len(_rows):
            g, used, kind = [i], {_rows[i]["channel"].value}, _rows[i]["kind"].value
            if kind != "delay":
                while (i + len(g) < len(_rows) and _rows[i + len(g)]["kind"].value == kind
                       and _rows[i + len(g)]["channel"].value not in used):
                    used.add(_rows[i + len(g)]["channel"].value)
                    g.append(i + len(g))
            groups.append(g)
            i += len(g)
        return groups

    def _time(text, name):
        text = (text or "").strip()
        if not text or text.lower() == "none":
            return None
        v = float(text)
        if v <= 0:
            raise ValueError(f"{name} must be a positive time in ms, or empty for none")
        return v

    def _state():
        """Everything a run needs, read from the widgets once."""
        T1 = [_time(s["T1"].value, f"T1 of spin {i}") for i, s in enumerate(_spins)]
        T2 = [_time(s["T2"].value, f"T2 of spin {i}") for i, s in enumerate(_spins)]
        return dict(nuclei=[s["nucleus"].value for s in _spins],
                    offsets=[s["offset"].value for s in _spins],
                    J={c["pair"]: c["J"].value for c in _couplings}, T1=T1, T2=T2,
                    durations=[r["duration"].value for r in _rows])

    def _system(st):
        J = {k: v for k, v in st["J"].items() if v != 0.0}
        ss = SpinSystem(nuclei=st["nuclei"], offsets=st["offsets"], couplings=J)
        rel = Relaxation(T1=st["T1"], T2=st["T2"])
        return ss, (None if rel.is_identity() else rel)

    def _segments(st):
        def shaped(r, dur):
            shape = RFShape.create(r["shape"].value, duration=dur,
                                   points=int(r["points"].value), time_reversed=bool(r["tr"].value))
            return Pulse(shape, np.deg2rad(r["flip"].value), axis=r["phase"].value,
                         backend=NumpyBackend(Gamma=gyro_ratio(r["channel"].value)))
        D = st["durations"]
        segs = []
        for g in _events():
            r = _rows[g[0]]
            k = r["kind"].value
            if k == "delay":
                segs.append(Delay(D[g[0]]))
            elif k == "ideal pulse":
                for s in g:
                    q = _rows[s]
                    segs.append(IdealPulse(np.deg2rad(q["flip"].value), phase=q["phase"].value,
                                           channel=q["channel"].value))
            elif len(g) == 1:
                segs.append(ShapePulseSegment(shaped(r, D[g[0]])))
            else:
                segs.append(SimultaneousPulses([shaped(_rows[s], D[s]) for s in g],
                                               align=r["align"].value))
                segs.extend(Delay(0.0) for _ in g[1:])
        return segs

    def _names():
        names = [n.strip() for n in w_read.value.split(",") if n.strip()]
        if not names:
            raise ValueError("read out is empty -- give product-operator names, e.g. 2IzSy")
        return names

    def _propagate(st, relax=True):
        ss, rel = _system(st)
        ops = SpinOperators(ss)
        terms = [t.strip() for t in w_start.value.split("+") if t.strip()]
        if not terms:
            raise ValueError("start is empty -- give a product-operator name such as Iz")
        sigma0 = sum(ops[t] for t in terms).astype(complex)
        sigma = LiouvilleSequence(_segments(st), ss,
                                  relaxation=rel if relax else None).propagate(sigma0)
        return {n: float(np.real(v)) for n, v in ops.readout(sigma, _names()).items()}

    # ---- fast sweeps: a pulse does not change when a delay does --------------------
    # Each finite pulse is mapped ONCE onto the product-operator basis: relaxation
    # pulls toward equilibrium, so the map is affine, c -> A c + b, found by
    # propagating every basis operator (and zero) through that pulse. A sweep then
    # propagates only its delays and applies the stored maps -- the same
    # propagation, segment by segment, as LiouvilleSequence.propagate.
    def _basis(n):
        import itertools, functools
        P1 = [np.eye(2), np.array([[0, 1], [1, 0]]), np.array([[0, -1j], [1j, 0]]),
              np.array([[1, 0], [0, -1]])]
        return np.array([functools.reduce(np.kron, c) for c in itertools.product(P1, repeat=n)],
                        dtype=complex)

    def _runner(st, targets, relax=True):
        """A function d -> {name: <name>} for the sequence with the target delay
        rows set to d; every finite pulse is propagated once, here."""
        ss, rel = _system(st)
        rel = rel if relax else None
        ops = SpinOperators(ss)
        terms = [t.strip() for t in w_start.value.split("+") if t.strip()]
        if not terms:
            raise ValueError("start is empty -- give a product-operator name such as Iz")
        sigma0 = sum(ops[t] for t in terms).astype(complex)
        names = _names()
        segs = _segments(st)
        P = _basis(ss.n_spins)
        dim = P.shape[1]
        coeff = lambda s: np.einsum("kij,ji->k", P, s) / dim
        prop = lambda seg, s: LiouvilleSequence([seg], ss, relaxation=rel).propagate(s)
        maps = {}
        for i, seg in enumerate(segs):
            if i in targets or isinstance(seg, (Delay, IdealPulse)):
                continue                     # delays and ideal pulses: cheap, run directly
            b = coeff(prop(seg, np.zeros((dim, dim), complex)))
            maps[i] = (np.stack([coeff(prop(seg, Pk)) - b for Pk in P], axis=1), b)

        def f(d):
            sigma = sigma0
            for i, seg in enumerate(segs):
                if i in targets:
                    sigma = prop(Delay(d), sigma)
                elif i in maps:
                    A, b = maps[i]
                    sigma = np.einsum("k,kij->ij", A @ coeff(sigma) + b, P)
                else:
                    sigma = prop(seg, sigma)
            return {n: float(np.real(v)) for n, v in ops.readout(sigma, names).items()}
        return f

    def _targets():
        sel = w_dsel.value
        if sel is None:
            raise ValueError("the sequence has no delay rows to sweep")
        if isinstance(sel, str) and sel.startswith("g:"):
            rows = [next(i for i, x in enumerate(_rows) if x is r) for r in _group_rows[sel[2:]]]
            return rows, PRESETS[w_preset.value]["sweep"][sel[2:]]
        if sel == "all":
            return [i for i, r in enumerate(_rows) if r["kind"].value == "delay"], None
        return [sel], None

    def _with_delay(st, targets, d):
        dur = list(st["durations"])
        for i in targets:
            dur[i] = d
        return {**st, "durations": dur}

    def _matches_preset():
        P = PRESETS[w_preset.value]
        if len(_spins) != len(P["spins"]) or len(_rows) != len(P["rows"]):
            return False
        for r, a in zip(_rows, P["rows"]):
            if r["kind"].value != a[0]:
                return False
            if a[0] != "delay" and (r["flip"].value, r["phase"].value, r["channel"].value) != (a[1], a[2], a[3]):
                return False
        return [s["nucleus"].value for s in _spins] == [s[0] for s in P["spins"]]

    def _formula(st, group):
        if group is None or group.get("formula") is None or not _matches_preset():
            return None
        f = group["formula"](st["J"], st["T1"], st["T2"], st["durations"])
        return f if f and f["op"] in _names() else None

    def _reference(st, group):
        if group is None or group.get("ref") is None:
            return None
        try:
            return group["ref"][0](st["J"], st["durations"]), group["ref"][1]
        except ZeroDivisionError:
            return None

    def _best(f, grid, name, vals=None):
        """Delay of largest |<name>|: on the grid (vals, if already computed), then
        refined between its neighbors. f: a _runner."""
        if vals is None:
            vals = np.array([f(d)[name] for d in grid])
        k = int(np.argmax(np.abs(vals)))
        if 0 < k < len(grid) - 1:
            sign = np.sign(vals[k]) or 1.0
            try:
                r = optimize_delay(lambda d: d, lambda d: sign * f(d)[name],
                                   (grid[k - 1], grid[k + 1]))
                return float(r.x), f(float(r.x))[name], vals
            except ValueError:
                pass
        return float(grid[k]), float(vals[k]), vals

    def _tex(name):
        m = re.fullmatch(r"(\d*)((?:[A-Za-z]'?[xyz])+)", name.replace(" ", ""))
        if m is None:
            return name
        coef, body = m.groups()
        parts = re.findall(r"([A-Za-z]'?)([xyz])", body)
        if not coef and len(parts) > 1:
            coef = str(2 ** (len(parts) - 1))
        return (r"$\langle " + ("" if coef in ("", "1") else coef)
                + "".join(f"{s}_{a}" for s, a in parts) + r" \rangle$")

    def _rel_text(st):
        parts = []
        for i, nuc in enumerate(st["nuclei"]):
            for t, v in (("T₁", st["T1"][i]), ("T₂", st["T2"][i])):
                if v is not None:
                    parts.append(f"{t}({nuc_label(nuc)}) {v:g} ms")
        return ", ".join(parts) if parts else "no relaxation"

    def _show_sequence(st):
        ss, _ = _system(st)
        fig, ax = plt.subplots(figsize=(FIG_W, 0.9 + 0.75 * len(ss.nuclei)))
        draw_sequence(LiouvilleSequence(_segments(st), ss), ax=ax, to_scale=False)
        fig.suptitle("the sequence as simulated")
        fig.tight_layout()
        plt.show()

    # ---- figures -------------------------------------------------------------------------
    def _fig_operators(st):
        a, b = _propagate(st), _propagate(st, relax=False)
        print(f"   {'operator':<10} {'relaxation':>12} {'none':>12}")
        for n in a:
            print(f"   {n:<10} {a[n]:>+12.6f} {b[n]:>+12.6f}")

    def _fig_delay(st):
        targets, group = _targets()
        which = ("rows " if len(targets) > 1 else "row ") + " + ".join(str(i + 1) for i in targets)
        names = _names()
        grid = np.linspace(w_dmin.value, w_dmax.value, int(w_dn.value))
        key = w_sweep.value
        t0 = time.time()
        if key == "none":
            runs = [("", st)]
        else:
            vals = parse_values(w_values.value)
            if not vals:
                raise ValueError("type the values to compare, e.g. 5, 10, 20")
            m = re.match(r"(T₂|T₁)\(([ISK])\)", key)
            runs = []
            for v in vals:
                s2 = {**st, "T1": list(st["T1"]), "T2": list(st["T2"]), "J": dict(st["J"])}
                if m:
                    s2["T2" if m.group(1) == "T₂" else "T1"]["ISK".index(m.group(2))] = v
                else:
                    i, j = map(int, re.findall(r"\d", key)[:2])
                    s2["J"][(i, j)] = v
                runs.append((f"{key.split(' (')[0]} = {v:g} {key.split('(')[-1][:-1]}", s2))
        shown = names if key == "none" else names[:1]
        curves = []
        for lab, s in runs:
            f = _runner(s, targets)
            res = {n: np.empty(len(grid)) for n in shown}
            for k, d in enumerate(grid):
                r = f(d)
                for n in shown:
                    res[n][k] = r[n]
            best = {n: _best(f, grid, n, res[n])[:2] for n in shown
                    if np.abs(res[n]).max() > 1e-6}
            curves.append((lab, s, res, best, f))
        f0 = _runner(st, targets, relax=False)
        r0 = [f0(d) for d in grid]
        none = {n: np.array([r[n] for r in r0]) for n in shown}
        print(f"  {len(runs)} run(s) x {len(grid)} values of {which} in {time.time() - t0:.1f} s")
        ref = _reference(st, group)
        for lab, s, res, best, f in curves:
            for n, (d_b, v_b) in best.items():
                txt = f"  {lab or _rel_text(s)}: <{n}> largest |value| {abs(v_b):.4f} at {d_b:.4f} ms"
                if ref and key == "none":
                    v_r = f(ref[0])[n]
                    txt += (f"; at {ref[1]} = {ref[0]:.4f} ms {abs(v_r):.4f} "
                            f"({100 * (abs(v_b) - abs(v_r)) / abs(v_b):.1f} % less)")
                print(txt)
        form = _formula(st, group) if key == "none" else None
        if form:
            print(f"  formula: best delay {form['opt']:.4f} ms")
        _show_sequence(st)
        fig, ax = plt.subplots(figsize=(FIG_W, 4))
        fig.suptitle("Delay")
        for k, (lab, s, res, best, _f) in enumerate(curves):
            for i, n in enumerate(shown):
                c = ("k" if len(shown) == 1 else f"C{i}") if key == "none" else PALETTE[k % len(PALETTE)]
                ax.plot(grid, res[n], color=c, lw=1.6, label=(_tex(n) if key == "none" else lab))
                if n in best:
                    ax.axvline(best[n][0], color=c, ls=":", lw=1)
        for i, n in enumerate(shown):
            c = "0.6" if len(shown) == 1 or key != "none" else f"C{i}"
            ax.plot(grid, none[n], color=c, lw=1.0, ls="--", alpha=0.8,
                    label=f"{_tex(n)}, no relaxation")
        if form:
            step = max(1, len(grid) // 25)
            ax.plot(grid[::step], form["curve"](grid[::step]), "o", ls="none", mfc="none", mec="C1",
                    ms=5, mew=1.2, label=form["label"])
            ax.axvline(form["opt"], color="C1", ls=":", lw=1,
                       label=f"{form['opt_label']} = {form['opt']:.4f} ms")
        if ref:
            ax.axvline(ref[0], color="k", ls="--", lw=1, label=f"{ref[1]} = {ref[0]:.4f} ms")
        ax.axhline(0, color="0.8", lw=0.5)
        ax.set(xlabel=f"delay: {which} (ms)", ylabel="expectation value", ylim=(-1.05, 1.05),
               yticks=[-1, -0.5, 0, 0.5, 1])
        ax.set_title(_rel_text(st) + ";  dotted: best delay of each curve", loc="left")
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            cols = {"delay (ms)": grid}
            for lab, s, res, best, f in curves:
                for n in shown:
                    cols[f"<{n}> [{lab}]" if lab else f"<{n}>"] = res[n]
            for n in shown:
                cols[f"<{n}> no relaxation"] = none[n]
            save_csv("pulsim_relaxation_delay", cols)

    def _fig_optimum(st):
        targets, group = _targets()
        which = ("rows " if len(targets) > 1 else "row ") + " + ".join(str(i + 1) for i in targets)
        name = _names()[0]
        i_spin = w_t2spin.value
        lo, hi, n = w_t2from.value, w_t2to.value, int(w_t2n.value)
        if not (0 < lo < hi) or n < 2:
            raise ValueError("T₂ range: 0 < from < to, and at least 2 points")
        T2s = np.geomspace(lo, hi, n)
        grid = np.linspace(max(w_dmin.value, 1e-3), w_dmax.value, 41)
        ref = _reference(st, group)
        lab_spin = f"T₂({nuc_label(st['nuclei'][i_spin])})"
        print(f"  <{name}>, {n} values of {lab_spin} from {lo:g} to {hi:g} ms "
              f"(each: 41 delays, then a refinement) ...")
        t0 = time.time()
        d_best, v_best, v_ref, f_opt, f_lab = [], [], [], [], None
        for T2 in T2s:
            s = {**st, "T2": list(st["T2"])}
            s["T2"][i_spin] = float(T2)
            f = _runner(s, targets)
            d_b, v_b, _ = _best(f, grid, name)
            d_best.append(d_b)
            v_best.append(abs(v_b))
            v_ref.append(abs(f(ref[0])[name]) if ref else np.nan)
            form = _formula(s, group)
            f_opt.append(form["opt"] if form and form["op"] == name else np.nan)
            f_lab = form["opt_label"] if form else f_lab
        d_best, v_best, v_ref, f_opt = map(np.array, (d_best, v_best, v_ref, f_opt))
        print(f"  done in {time.time() - t0:.1f} s")
        _show_sequence(st)
        fig, axs = plt.subplots(1, 2, figsize=(FIG_W, 3.0))
        a = axs[0]
        a.plot(T2s, d_best, color="k", lw=1.6, marker="o", ms=3, label="best delay")
        if np.isfinite(f_opt).all():
            a.plot(T2s, f_opt, color="C1", lw=1.0, ls="--", label=f_lab)
        if ref:
            a.axhline(ref[0], color="0.5", ls=":", lw=1, label=f"{ref[1]} = {ref[0]:.4f} ms")
        a.set(xscale="log", xlabel=f"{lab_spin} (ms)", ylabel=f"best delay: {which} (ms)")
        a.legend(loc="lower right", frameon=False)
        a = axs[1]
        a.plot(T2s, v_best, color="k", lw=1.6, marker="o", ms=3, label="at the best delay")
        if ref:
            a.plot(T2s, v_ref, color="0.5", lw=1.2, ls="--", label=f"at {ref[1]}")
        a.set(xscale="log", xlabel=f"{lab_spin} (ms)", ylabel=f"|{_tex(name)}|", ylim=(0, 1.05))
        a.legend(loc="lower right", frameon=False)
        fig.suptitle(f"{w_preset.value}: best delay against {lab_spin}")
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            cols = {f"T2 of spin {i_spin} (ms)": T2s, "best delay (ms)": d_best,
                    "|value| at best": v_best}
            if ref:
                cols[f"|value| at {ref[1]}"] = v_ref
            if np.isfinite(f_opt).all():
                cols["formula best delay (ms)"] = f_opt
            save_csv("pulsim_relaxation_optimum", cols)

    FIG_FUNCS = {"Delay optimization": _fig_delay, "Optimum vs T₂": _fig_optimum,
                 "Product operators": _fig_operators}

    def _run(_b):
        with out:
            clear_output(wait=True)
            try:
                print("Run button has been clicked. Now running... Please wait...")
                st = _state()
                print(f"{len(_rows)} segments on {', '.join(nuc_label(n) for n in st['nuclei'])}   "
                      f"start = {w_start.value}   {_rel_text(st)}   figure = {w_figure.value}")
                stem = "pulsim_relaxation_" + w_figure.value.split()[0].lower()
                with save_figures(w_figsave.value, stem):
                    if w_figure.value == "Product operators":
                        _show_sequence(st)
                    FIG_FUNCS[w_figure.value](st)
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")
            finally:
                flush_downloads()      # last: nothing may print after a Colab download

    b_add   = widgets.Button(description="+ Add segment", layout=W("130px"))
    b_run   = widgets.Button(description="Run", button_style="primary", layout=W("110px"))
    b_reset = widgets.Button(description="Reload preset", layout=W("120px"),
                             tooltip="restore the selected preset (undo your edits)")
    b_add.on_click(_add)
    b_run.on_click(_run)
    b_reset.on_click(lambda _b: _load_preset(_b))
    _rebuild_system()
    _load_preset()

    header = widgets.HBox([L("#", "26px"), L("segment", "120px"), L("flip °", "75px"),
                           L("phase", "60px"), L("dur ms", "80px"), L("shape", "130px"),
                           L("points", "75px"), L("TR", "40px"), L("channel", "80px"),
                           L("align", "80px")])
    display(widgets.VBox([
        widgets.HBox([w_preset, preset_note]),
        separator(),
        widgets.HTML("<b>Spin system</b> — offsets in rad/ms, couplings in Hz, relaxation "
                     "times in ms (empty = none on that axis)"),
        w_nspins,
        widgets.HBox([L("", "90px"), L("nucleus", "90px"), L("offset", "90px"),
                      L("T₁ (ms)", "90px"), L("T₂ (ms)", "90px")]),
        spin_box, coupling_box,
        separator(),
        widgets.HTML("<b>Segments</b> — applied in order; TR = time-reversed shaped pulse.<br>"
                     "Consecutive pulses on different channels play simultaneously (align: "
                     "left / center / right, set on the group's first row)."),
        header, seg_box,
        separator(),
        widgets.HTML("<b>Figure</b> — start and read out are product-operator names; the "
                     "best delay is that of the first read-out operator"),
        widgets.HBox([w_figure, w_start, w_read]),
        delay_box, compare_box, t2_box,
        separator(),
        widgets.HBox([b_add, b_run, b_reset, w_figsave, w_data]),
        out,
    ]))


relaxation_widget()

In [ ]:
#@title 4. Coherence selection with gradients { display-mode: "form" }
from PULSIM.spin_operators import SpinOperators, Iz, embed
from PULSIM.liouville import IdealPulse, SimultaneousPulses


def coherence_widget():
    import re
    PI = np.pi
    NUCLEI = ["H", "13C", "15N", "19F", "31P"]
    NUC_OPTS = [(nuc_label(n), n) for n in NUCLEI]
    SEG_KINDS = ["ideal pulse", "delay", "shaped pulse", "gradient"]
    PHASES = ["x", "y", "-x", "-y"]
    ALIGNS = ["left", "center", "right"]
    SHAPES = sorted(s for s in RFShape.available() if s not in ("file", "composite", "varian"))
    GRAD_SHAPES = ["sine", "rectangular"]
    GRAD_AREA = {"sine": 2.0 / np.pi, "rectangular": 1.0}      # mean amplitude / peak
    FIGURES = ["Product operators", "Coherence pathways", "Gradient ratio"]

    # ---- presets (rows: kind, flip, phase, channel, ms[, options]) --------------------
    def G(dur, g, shape="sine"):
        return ("gradient", 0.0, "x", "H", dur, dict(shape=shape, g=g))

    def gamma_ratio(a, b):
        return gyro_ratio(a) / gyro_ratio(b)

    def first_g(st):
        return next(r["g"] for r in st["rows"] if r["kind"] == "gradient")

    def reverse_inept(S, J, g2):
        d = 1 / (4 * J / 1000)
        return [G(1.0, 80.0),
                ("ideal pulse", 90.0, "x", "H", 0.0), ("ideal pulse", 90.0, "x", S, 0.0),
                ("delay", 0.0, "x", "H", d),
                ("ideal pulse", 180.0, "x", "H", 0.0), ("ideal pulse", 180.0, "x", S, 0.0),
                ("delay", 0.0, "x", "H", d), G(1.0, g2)]

    PRESETS = {
        "Gradient echo (1H)": dict(
            spins=[("H", 0.0)], J={},
            rows=[("ideal pulse", 90.0, "x", "H", 0.0), G(1.0, 40.0),
                  ("ideal pulse", 180.0, "x", "H", 0.0), G(1.0, 40.0)],
            start="Iz", read="Ix, Iy", gsweep=(-100.0, 100.0),
            expect=lambda st: [(first_g(st), "G2 = G1")],
            note="G1 winds a helix of phase along the sample, the 180° reverses it and G2 "
                 "unwinds it: the signal returns only when G2 = G1. Remove the 180° and it "
                 "returns at G2 = −G1."),
        "Echo/antiecho (1H-15N)": dict(
            spins=[("H", 0.0), ("15N", 0.0)], J={(0, 1): 92.0},
            rows=reverse_inept("15N", 92.0, -80.0 * abs(gamma_ratio("15N", "H"))),
            start="2IzSy", read="Ix, Iy", gsweep=(-20.0, 20.0),
            expect=lambda st: [(s * first_g(st) * abs(gamma_ratio("15N", "H")), l)
                               for s, l in ((-1, "−G1·|γN/γH|"), (1, "+G1·|γN/γH|"))],
            note="G1 encodes the 15N coherence (after INEPT), the reverse INEPT moves it to "
                 "1H, G2 decodes: G2/G1 = ±γN/γH (8.1 % for 80 %) brings back one of the "
                 "two pathways, echo or antiecho — half the signal. J also evolves during "
                 "the gradients."),
        "Echo/antiecho (1H-13C)": dict(
            spins=[("H", 0.0), ("13C", 0.0)], J={(0, 1): 140.0},
            rows=reverse_inept("13C", 140.0, 80.0 * abs(gamma_ratio("13C", "H"))),
            start="2IzSy", read="Ix, Iy", gsweep=(-40.0, 40.0),
            expect=lambda st: [(s * first_g(st) * abs(gamma_ratio("13C", "H")), l)
                               for s, l in ((-1, "−G1·|γC/γH|"), (1, "+G1·|γC/γH|"))],
            note="As for 15N, with γC/γH ≈ 0.25: G2 = ±20 % for G1 = 80 %."),
        "Double-quantum filter (1H-1H)": dict(
            spins=[("H", 0.0), ("H", 0.0)], J={(0, 1): 7.0},
            rows=[("ideal pulse", 90.0, "x", "H", 0.0), G(1.0, 20.0),
                  ("ideal pulse", 90.0, "x", "H", 0.0), G(1.0, 40.0)],
            start="2IxSz", read="2IxSz, 2IzSx", gsweep=(-60.0, 60.0),
            expect=lambda st: [(s * 2 * first_g(st), l) for s, l in ((-1, "G2 = −2 G1"),
                                                                      (1, "G2 = 2 G1"))],
            note="The first 90° turns the antiphase 2IxSz partly into double-quantum "
                 "coherence (p = ±2); G2 = ±2 G1 keeps only the path through p = ±2: the "
                 "double-quantum filter of DQF-COSY, a quarter on each spin."),
        "Build your own": dict(
            spins=[("H", 0.0), ("13C", 0.0)], J={(0, 1): 140.0},
            rows=[("ideal pulse", 90.0, "x", "H", 0.0), G(1.0, 40.0)],
            start="Iz", read="Ix, Iy", gsweep=(-100.0, 100.0), expect=lambda st: [],
            note="Add pulses, delays and gradients with + Add segment."),
    }
    DEFAULTS = dict(gmax=50.0, sample=1.5, gn=401, figsave="no", data="no")

    S_ = {"description_width": "initial"}
    def W(px): return widgets.Layout(width=px)
    def L(text, px): return widgets.Label(text, layout=W(px))
    out = widgets.Output()

    # ---- spin system ----------------------------------------------------------------
    w_nspins = widgets.IntSlider(value=2, min=1, max=3, description="spins:", style=S_,
                                 layout=W("200px"))
    spin_box, coupling_box = widgets.VBox([]), widgets.VBox([])
    _spins, _couplings = [], []

    def _rebuild_system(change=None):
        n = w_nspins.value
        while len(_spins) < n:
            i = len(_spins)
            _spins.append({
                "nucleus": widgets.Dropdown(options=NUC_OPTS, value="H" if i == 0 else "13C",
                                            layout=W("90px")),
                "offset": widgets.FloatText(value=0.0, layout=W("90px"))})
        del _spins[n:]
        spin_box.children = tuple(
            widgets.HBox([L(f"spin {i} ({'ISK'[i]})", "90px"), s["nucleus"], s["offset"]])
            for i, s in enumerate(_spins))
        pairs = [(i, j) for i in range(n) for j in range(i + 1, n)]
        old = {p["pair"]: p["J"].value for p in _couplings}
        _couplings.clear()
        for (i, j) in pairs:
            _couplings.append({"pair": (i, j),
                               "J": widgets.FloatText(value=old.get((i, j), 0.0), layout=W("90px"))})
        coupling_box.children = tuple(widgets.HBox([L(f"J({i},{j}) Hz", "90px"), c["J"]])
                                      for c, (i, j) in zip(_couplings, pairs))

    w_nspins.observe(_rebuild_system, names="value")

    # ---- figure and gradient controls ------------------------------------------------
    w_figure = widgets.Dropdown(options=FIGURES, description="figure:", style=S_, layout=W("230px"))
    w_start  = widgets.Text(description="start:", style=S_, layout=W("160px"))
    w_read   = widgets.Text(description="read out:", style=S_, layout=W("260px"))
    w_gmax   = widgets.FloatText(description="G max (G/cm):", style=S_, layout=W("160px"),
                                 tooltip="the gradient at 100 %")
    w_sample = widgets.FloatText(description="sample length (cm):", style=S_, layout=W("190px"),
                                 tooltip="length of the sample along the gradient (the part "
                                         "the coil sees), centered on the gradient")
    w_gsel   = widgets.Dropdown(description="sweep:", style=S_, layout=W("200px"))
    w_gfrom  = widgets.FloatText(description="from (%):", style=S_, layout=W("140px"))
    w_gto    = widgets.FloatText(description="to (%):", style=S_, layout=W("130px"))
    w_gn     = widgets.IntText(description="steps:", style=S_, layout=W("120px"))
    w_figsave = widgets.Dropdown(options=FIG_FORMATS, description="figures:", style=S_,
                                 layout=W("200px"))
    w_data   = widgets.Dropdown(options=DATA_FORMATS, description="data:", style=S_,
                                layout=W("140px"))
    ratio_box = widgets.HBox([w_gsel, w_gfrom, w_gto, w_gn])

    def _show_figure_controls(change=None):
        ratio_box.layout.display = "" if w_figure.value == "Gradient ratio" else "none"
    w_figure.observe(_show_figure_controls, names="value")

    # ---- one row per segment (module 3's rows, plus gradients) ------------------------
    _rows = []
    seg_box = widgets.VBox([])
    w_preset = widgets.Dropdown(options=list(PRESETS), description="preset:", style=S_,
                                layout=W("270px"))
    preset_note = widgets.HTML()

    def _sync_gsel():
        opts = [(f"row {i + 1}", i) for i, r in enumerate(_rows) if r["kind"].value == "gradient"]
        old = w_gsel.value
        w_gsel.options = opts or [("(no gradient rows)", None)]
        vals = [v for _, v in w_gsel.options]
        w_gsel.value = old if old in vals else vals[-1]          # default: the last gradient

    def _refresh():
        for n, r in enumerate(_rows, start=1):
            r["idx"].value = str(n)
            r["up"].disabled = (n == 1)
            r["down"].disabled = (n == len(_rows))
        seg_box.children = tuple(r["box"] for r in _rows)
        _sync_gsel()

    def _move(row, step):
        i = _rows.index(row)
        j = i + step
        if 0 <= j < len(_rows):
            _rows[i], _rows[j] = _rows[j], _rows[i]
            _refresh()

    def _make_row(kind="ideal pulse", flip=90.0, phase="x", channel="H", duration=1.0,
                  shape="gausscasq5", points=500, tr=False, align="center", g=40.0):
        idx = L("", "26px")
        w_kind = widgets.Dropdown(options=SEG_KINDS, value=kind, layout=W("120px"))
        w_flip = widgets.FloatText(value=flip, layout=W("75px"))
        w_ph   = widgets.Dropdown(options=PHASES, value=phase, layout=W("60px"))
        w_ch   = widgets.Dropdown(options=NUC_OPTS, value=channel, layout=W("80px"))
        w_dur  = widgets.FloatText(value=round(duration, 6), layout=W("80px"))
        opts = GRAD_SHAPES if kind == "gradient" else SHAPES
        w_shp  = widgets.Dropdown(options=opts, value=shape if shape in opts else opts[0],
                                  layout=W("130px"))
        w_pts  = widgets.IntText(value=points, layout=W("75px"))
        w_tr   = widgets.Checkbox(value=tr, indent=False, layout=W("40px"),
                                  tooltip="time-reversed: play the shape backwards (shaped pulses)")
        w_al   = widgets.Dropdown(options=ALIGNS, value=align, layout=W("80px"),
                                  tooltip="shaped pulses played together: where a shorter one "
                                          "sits inside the longest (the first row of the group decides)")
        w_g    = widgets.FloatText(value=g, layout=W("70px"),
                                   tooltip="gradient strength, % of G max (negative: reversed)")
        w_up   = widgets.Button(description="↑", tooltip="move this segment up", layout=W("32px"))
        w_dn   = widgets.Button(description="↓", tooltip="move this segment down", layout=W("32px"))
        w_rm   = widgets.Button(description="✕", tooltip="remove", layout=W("36px"))

        def _sync(change=None):
            k = w_kind.value
            want = GRAD_SHAPES if k == "gradient" else SHAPES
            if tuple(w_shp.options) != tuple(want):
                w_shp.options = want
                w_shp.value = "sine" if k == "gradient" else "gausscasq5"
            w_flip.disabled = w_ph.disabled = w_ch.disabled = k in ("delay", "gradient")
            w_pts.disabled = w_tr.disabled = w_al.disabled = (k != "shaped pulse")
            w_shp.disabled = k not in ("shaped pulse", "gradient")
            w_dur.disabled = (k == "ideal pulse")
            w_g.disabled = (k != "gradient")
            if change is not None:
                _sync_gsel()
        w_kind.observe(_sync, names="value")
        _sync()

        def _remove(_b):
            if len(_rows) == 1:
                with out:
                    clear_output(wait=True)
                    print("A sequence needs at least one segment.")
                return
            _rows.remove(row)
            _refresh()
        w_rm.on_click(_remove)
        w_up.on_click(lambda _b: _move(row, -1))
        w_dn.on_click(lambda _b: _move(row, +1))
        row = {"idx": idx, "kind": w_kind, "flip": w_flip, "phase": w_ph, "channel": w_ch,
               "duration": w_dur, "shape": w_shp, "points": w_pts, "tr": w_tr, "align": w_al,
               "g": w_g, "up": w_up, "down": w_dn,
               "box": widgets.HBox([idx, w_kind, w_flip, w_ph, w_dur, w_shp, w_pts, w_tr,
                                    w_ch, w_al, w_g, w_up, w_dn, w_rm])}
        return row

    def _add(_b=None):
        if _rows:
            p = _rows[-1]
            _rows.append(_make_row(p["kind"].value, p["flip"].value, p["phase"].value,
                                   p["channel"].value, p["duration"].value, p["shape"].value,
                                   p["points"].value, p["tr"].value, p["align"].value,
                                   p["g"].value))
        else:
            _rows.append(_make_row())
        _refresh()

    def _load_preset(change=None):
        p, d = PRESETS[w_preset.value], DEFAULTS
        w_nspins.value = len(p["spins"])
        _rebuild_system()
        for s_, (nuc, off) in zip(_spins, p["spins"]):
            s_["nucleus"].value, s_["offset"].value = nuc, off
        for c in _couplings:
            c["J"].value = p["J"].get(c["pair"], 0.0)
        _rows.clear()
        for a in p["rows"]:
            _rows.append(_make_row(*a[:5], **(a[5] if len(a) > 5 else {})))
        _refresh()
        w_start.value, w_read.value = p["start"], p["read"]
        w_gmax.value, w_sample.value = d["gmax"], d["sample"]
        w_gfrom.value, w_gto.value, w_gn.value = p["gsweep"][0], p["gsweep"][1], d["gn"]
        w_figsave.value, w_data.value = d["figsave"], d["data"]
        _show_figure_controls()
        preset_note.value = f"<i>{p['note']}</i>"
        if change is not None:
            with out:
                clear_output(wait=True)
                print(f"Loaded preset: {w_preset.value}")

    w_preset.observe(_load_preset, names="value")

    # ---- the sequence ------------------------------------------------------------------
    def _events():
        """Consecutive pulses of the same kind on different channels play together;
        delays and gradients are events of their own."""
        groups, i = [], 0
        while i < len(_rows):
            g, used, kind = [i], {_rows[i]["channel"].value}, _rows[i]["kind"].value
            if kind not in ("delay", "gradient"):
                while (i + len(g) < len(_rows) and _rows[i + len(g)]["kind"].value == kind
                       and _rows[i + len(g)]["channel"].value not in used):
                    used.add(_rows[i + len(g)]["channel"].value)
                    g.append(i + len(g))
            groups.append(g)
            i += len(g)
        return groups

    def _state():
        return dict(nuclei=[s["nucleus"].value for s in _spins],
                    offsets=[s["offset"].value for s in _spins],
                    J={c["pair"]: c["J"].value for c in _couplings},
                    rows=[dict(kind=r["kind"].value, dur=r["duration"].value, g=r["g"].value,
                               shape=r["shape"].value) for r in _rows],
                    gmax=w_gmax.value, sample=w_sample.value)

    def _system(st):
        J = {k: v for k, v in st["J"].items() if v != 0.0}
        return SpinSystem(nuclei=st["nuclei"], offsets=st["offsets"], couplings=J)

    def _segments(st):
        """One segment per row (index = row); a gradient row is a Delay here and
        gets its phase in _propagate."""
        def shaped(r):
            shape = RFShape.create(r["shape"].value, duration=r["duration"].value,
                                   points=int(r["points"].value), time_reversed=bool(r["tr"].value))
            return Pulse(shape, np.deg2rad(r["flip"].value), axis=r["phase"].value,
                         backend=NumpyBackend(Gamma=gyro_ratio(r["channel"].value)))
        segs = []
        for g in _events():
            r = _rows[g[0]]
            k = r["kind"].value
            if k in ("delay", "gradient"):
                segs.append(Delay(st["rows"][g[0]]["dur"]))
            elif k == "ideal pulse":
                for s in g:
                    q = _rows[s]
                    segs.append(IdealPulse(np.deg2rad(q["flip"].value), phase=q["phase"].value,
                                           channel=q["channel"].value))
            elif len(g) == 1:
                segs.append(ShapePulseSegment(shaped(r)))
            else:
                segs.append(SimultaneousPulses([shaped(_rows[s]) for s in g], align=r["align"].value))
                segs.extend(Delay(0.0) for _ in g[1:])
        return segs

    def _names():
        names = [n.strip() for n in w_read.value.split(",") if n.strip()]
        if not names:
            raise ValueError("read out is empty -- give product-operator names, e.g. Ix")
        return names

    def _turns_matrix(st, ss, r):
        """Turns of phase across the sample that a gradient row gives each element
        |a><b| of the density matrix: sum over spins of gamma_i * p_i, with
        p_i = m_i(a) - m_i(b) that spin's coherence order."""
        n = ss.n_spins
        m = np.array([np.real(np.diag(embed(Iz(), i, n))) for i in range(n)])     # (n, dim)
        p = m[:, :, None] - m[:, None, :]                                       # (n, dim, dim)
        gam = np.array([gyro_ratio(x) for x in st["nuclei"]])                   # kHz/mT
        G = r["g"] / 100.0 * st["gmax"] * 10.0 * GRAD_AREA[r["shape"]]          # mean mT/m
        return np.einsum("i,ijk->jk", gam, p) * G * (st["sample"] / 100.0) * r["dur"]

    def _start(ss):
        ops = SpinOperators(ss)
        terms = [t.strip() for t in w_start.value.split("+") if t.strip()]
        if not terms:
            raise ValueError("start is empty -- give a product-operator name such as Iz")
        return ops, sum(ops[t] for t in terms).astype(complex)

    def _advance(paths, segs, ss, st, rows, gradients=True):
        """Carry the coherence pathways {K: sigma_K} through rows `rows`. A gradient
        splits every sigma_K by the turns each element gathers (K -> K + T);
        chemical shift and J evolve during it as during a delay."""
        chunk = []
        def apply():
            if chunk:
                seq = LiouvilleSequence(list(chunk), ss)
                for K in list(paths):
                    paths[K] = seq.propagate(paths[K])
                chunk.clear()
        for i in rows:
            chunk.append(segs[i])
            r = st["rows"][i]
            if r["kind"] == "gradient" and gradients:
                apply()
                T = np.round(_turns_matrix(st, ss, r), 6)
                new = {}
                for K, s in paths.items():
                    for t in np.unique(T):
                        w = np.where(T == t, s, 0)
                        if np.abs(w).max() < 1e-14:
                            continue
                        k2 = round(K + float(t), 6)
                        new[k2] = new[k2] + w if k2 in new else w
                paths.clear()
                paths.update(new)
        apply()
        return paths

    def _readout(ops, paths):
        sigma = sum(s * np.sinc(K) for K, s in paths.items())     # average over the sample
        return {n: float(np.real(v)) for n, v in ops.readout(sigma, _names()).items()}

    def _propagate(st, gradients=True):
        ss = _system(st)
        ops, s0 = _start(ss)
        paths = _advance({0.0: s0}, _segments(st), ss, st, range(len(st["rows"])), gradients)
        return _readout(ops, paths), paths

    def _show_sequence(st):
        ss = _system(st)
        rows = st["rows"]
        gmax_abs = max([abs(r["g"]) for r in rows if r["kind"] == "gradient"] or [1.0]) or 1.0
        grads = [(i, r["g"] / gmax_abs, r["shape"]) for i, r in enumerate(rows)
                 if r["kind"] == "gradient"]
        show_segments(_segments(st), ss, to_scale=False, gradients=grads)

    def _tex(name):
        m = re.fullmatch(r"(\d*)((?:[A-Za-z]'?[xyz])+)", name.replace(" ", ""))
        if m is None:
            return name
        coef, body = m.groups()
        parts = re.findall(r"([A-Za-z]'?)([xyz])", body)
        if not coef and len(parts) > 1:
            coef = str(2 ** (len(parts) - 1))
        return (r"$\langle " + ("" if coef in ("", "1") else coef)
                + "".join(f"{s}_{a}" for s, a in parts) + r" \rangle$")

    # ---- figures -------------------------------------------------------------------------
    def _print_gradients(st):
        ss = _system(st)
        for i, r in enumerate(st["rows"]):
            if r["kind"] == "gradient":
                T = _turns_matrix(st, ss, r)
                per = ", ".join(f"{nuc_label(x)} {gyro_ratio(x) * r['g'] / 100 * st['gmax'] * 10 * GRAD_AREA[r['shape']] * st['sample'] / 100 * r['dur']:+.1f}"
                                for x in dict.fromkeys(st["nuclei"]))
                print(f"  row {i + 1}: gradient {r['g']:g} % of {st['gmax']:g} G/cm, {r['shape']}, "
                      f"{r['dur']:g} ms -> turns across {st['sample']:g} cm per unit coherence "
                      f"order: {per}")

    def _fig_operators(st):
        a, paths = _propagate(st)
        b, _ = _propagate(st, gradients=False)
        print(f"   {'operator':<10} {'gradients':>12} {'none':>12}")
        for n in a:
            print(f"   {n:<10} {a[n]:>+12.6f} {b[n]:>+12.6f}")
        kept = sorted({round(K, 3) for K in paths if abs(np.sinc(K)) > 0.05}, key=abs)
        print("  pathways that survive the average (K = turns left across the sample, "
              "weight sin(pi K)/(pi K) > 0.05): "
              + (", ".join(f"K = {K:+.3g} ({np.sinc(K):+.3f})" for K in kept) or "none"))

    def _fig_pathways(st):
        ss = _system(st)
        ops, s0 = _start(ss)
        paths = _advance({0.0: s0}, _segments(st), ss, st, range(len(st["rows"])))
        names = _names()
        amp = {K: {n: round(float(np.real(v)), 6) + 0.0 for n, v in ops.readout(s, names).items()}
               for K, s in paths.items()}
        amp = {K: a for K, a in amp.items() if max(abs(x) for x in a.values()) > 1e-3}
        Ks = np.array(sorted(amp))
        avg = {n: round(sum(amp[K][n] * np.sinc(K) for K in Ks), 4) + 0.0 for n in names}
        print(f"   {'K':>10} {'weight':>8} " + " ".join(f"{n:>10}" for n in names))
        for K in Ks:
            print(f"   {K:>+10.3f} {np.sinc(K):>+8.3f} "
                  + " ".join(f"{amp[K][n]:>+10.3f}" for n in names))
        print("   sample average (sum of value x weight): "
              + ", ".join(f"{n} {avg[n]:+.3f}" for n in names))
        _show_sequence(st)
        kmax = 1.1 * max([abs(K) for K in Ks] + [4.0])
        k = np.linspace(-kmax, kmax, 4001)
        fig, ax = plt.subplots(figsize=(FIG_W, 3.0))
        ax.plot(k, np.sinc(k), color="0.75", lw=0.8, label=r"weight $\sin(\pi K)/(\pi K)$")
        for j, n in enumerate(names):
            v = np.array([amp[K][n] for K in Ks])
            keep = np.abs(v) > 1e-3
            dx = (j - (len(names) - 1) / 2) * 0.008 * kmax       # side by side at one K
            c = "k" if len(names) == 1 else f"C{j}"
            ax.vlines(Ks[keep] + dx, 0, v[keep], color=c, lw=1.5)
            ax.plot(Ks[keep] + dx, v[keep], "o", color=c, ms=3,
                    label=f"{_tex(n)}  (average {avg[n]:+.3f})")
        ax.axhline(0, color="0.85", lw=0.5)
        ax.set(xlabel="K = turns of phase left across the sample", ylabel="expectation value",
               xlim=(-kmax, kmax), ylim=(-1.05, 1.05), yticks=[-1, -0.5, 0, 0.5, 1])
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
        fig.suptitle("Coherence pathways")
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            save_csv("pulsim_coherence_pathways",
                     {"K (turns)": Ks, "weight": np.sinc(Ks),
                      **{f"<{n}>": np.array([amp[K][n] for K in Ks]) for n in names}})

    def _fig_ratio(st):
        i_g = w_gsel.value
        if i_g is None:
            raise ValueError("the sequence has no gradient rows to sweep")
        ss = _system(st)
        ops, s0 = _start(ss)
        segs = _segments(st)
        names = _names()
        before = _advance({0.0: s0}, segs, ss, st, range(i_g))     # everything before it, once
        vals = np.linspace(w_gfrom.value, w_gto.value, int(w_gn.value))
        t0 = time.time()
        res = {n: np.empty(len(vals)) for n in names}
        for k, v in enumerate(vals):
            st2 = {**st, "rows": [dict(r) for r in st["rows"]]}
            st2["rows"][i_g]["g"] = v
            paths = _advance({K: s.copy() for K, s in before.items()}, segs, ss, st2,
                             range(i_g, len(st["rows"])))
            r = _readout(ops, paths)
            for n in names:
                res[n][k] = r[n]
        print(f"  {len(vals)} strengths of row {i_g + 1} in {time.time() - t0:.1f} s")
        size = np.sqrt(sum(res[n] ** 2 for n in names))
        peaks = [k for k in range(1, len(vals) - 1)          # main peaks, not the sinc wiggles
                 if size[k] >= size[k - 1] and size[k] >= size[k + 1]
                 and size[k] > max(0.02, 0.3 * size.max())]
        print("  the signal comes back at: " + (", ".join(f"{vals[k]:+.2f} % ({size[k]:.3f})"
                                                       for k in peaks) or "nowhere in this range"))
        expect = PRESETS[w_preset.value]["expect"](st) if _matches_preset() else []
        _show_sequence(st)
        fig, ax = plt.subplots(figsize=(FIG_W, 3.4))
        for j, n in enumerate(names):
            ax.plot(vals, res[n], lw=1.4, color="k" if len(names) == 1 else f"C{j}", label=_tex(n))
        for v, lab in expect:
            ax.axvline(v, color="0.4", ls=":", lw=1, label=f"{lab} = {v:+.2f} %")
        ax.axhline(0, color="0.8", lw=0.5)
        ax.set(xlabel=f"row {i_g + 1}: gradient strength (% of {st['gmax']:g} G/cm)",
               ylabel="expectation value", ylim=(-1.05, 1.05), yticks=[-1, -0.5, 0, 0.5, 1],
               xlim=(vals[0], vals[-1]))
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
        fig.suptitle("Gradient ratio")
        fig.tight_layout()
        plt.show()
        if w_data.value == "CSV":
            save_csv("pulsim_coherence_ratio", {f"row {i_g + 1} G (%)": vals,
                                                **{f"<{n}>": res[n] for n in names}})

    def _matches_preset():
        P = PRESETS[w_preset.value]
        if len(_rows) != len(P["rows"]) or [s["nucleus"].value for s in _spins] != [s[0] for s in P["spins"]]:
            return False
        return all(r["kind"].value == a[0] for r, a in zip(_rows, P["rows"]))

    def _run(_b):
        with out:
            clear_output(wait=True)
            try:
                print("Run button has been clicked. Now running... Please wait...")
                st = _state()
                print(f"{len(_rows)} segments on {', '.join(nuc_label(n) for n in st['nuclei'])}   "
                      f"start = {w_start.value}   figure = {w_figure.value}")
                _print_gradients(st)
                stem = "pulsim_coherence_" + {"Product operators": "operators",
                                          "Coherence pathways": "pathways",
                                          "Gradient ratio": "ratio"}[w_figure.value]
                with save_figures(w_figsave.value, stem):
                    if w_figure.value == "Product operators":
                        _show_sequence(st)
                        _fig_operators(st)
                    elif w_figure.value == "Coherence pathways":
                        _fig_pathways(st)
                    else:
                        _fig_ratio(st)
            except Exception as exc:
                print(f"{type(exc).__name__}: {exc}")
            finally:
                flush_downloads()      # last: nothing may print after a Colab download

    b_add   = widgets.Button(description="+ Add segment", layout=W("130px"))
    b_run   = widgets.Button(description="Run", button_style="primary", layout=W("110px"))
    b_reset = widgets.Button(description="Reload preset", layout=W("120px"),
                             tooltip="restore the selected preset (undo your edits)")
    b_add.on_click(_add)
    b_run.on_click(_run)
    b_reset.on_click(lambda _b: _load_preset(_b))
    _rebuild_system()
    _load_preset()

    header = widgets.HBox([L("#", "26px"), L("segment", "120px"), L("flip °", "75px"),
                           L("phase", "60px"), L("dur ms", "80px"), L("shape", "130px"),
                           L("points", "75px"), L("TR", "40px"), L("channel", "80px"),
                           L("align", "80px"), L("G %", "70px")])
    display(widgets.VBox([
        widgets.HBox([w_preset, preset_note]),
        separator(),
        widgets.HTML("<b>Spin system</b> — offsets in rad/ms, couplings in Hz"),
        w_nspins, spin_box, coupling_box,
        widgets.HBox([w_gmax, w_sample]),
        separator(),
        widgets.HTML("<b>Segments</b> — applied in order; a gradient row takes a shape (sine "
                     "or rectangular) and G % of G max; consecutive pulses on different "
                     "channels play simultaneously"),
        header, seg_box,
        separator(),
        widgets.HTML("<b>Figure</b> — start and read out are product-operator names"),
        widgets.HBox([w_figure, w_start, w_read]),
        ratio_box,
        separator(),
        widgets.HBox([b_add, b_run, b_reset, w_figsave, w_data]),
        out,
    ]))


coherence_widget()